# Code With SUZH

## Python From First Principles

### Advanced Python

**By SUZH TEAM**
**Presented By Muhammad Usman Rouf, CEO**

---

This is the third volume. Basics gave you the language's surface; Intermediate gave you objects, iteration, and the tools of a working developer. This volume is about the machinery underneath all of it — why an attribute lookup finds what it finds, what a coroutine actually is while it's suspended, what the GIL does and doesn't prevent, where a program's time actually goes once you measure it instead of guessing.

None of what follows is trivia. Every chapter exists because it changes what you can reason about correctly. A separate practice workbook exists for drilling exercises; this book is for building the mental models that make those exercises make sense.

Python 3.11+ throughout. Where a behavior is specific to CPython — the interpreter you're almost certainly running — rather than guaranteed by the Python language itself, that distinction will be called out explicitly, because conflating the two is a common source of confusion once you start reasoning about performance and concurrency.

# Part I — Advanced Object Model

## 61. Python's Data Model

Start with something you've written thousands of times and never had a reason to question: 

In [ ]:
a = 3
b = 4
print(a + b)

`a + b` looks like a language primitive — as though `+` were wired directly into Python the way it might be in a calculator. It isn't. Watch what happens with a type that isn't a number at all: 

In [ ]:
print("cat" + "dog")
print([1, 2] + [3, 4])

`+` did something completely different each time — numeric addition, string concatenation, list concatenation — and yet it's the same operator, the same syntax, applied by the same interpreter. Python is not special-casing each type inside the implementation of `+`. It's asking each object, uniformly, the same question: *do you know how to be added to something?* This is Python's **data model**: a fixed set of operations — arithmetic, comparison, length, indexing, iteration, calling — each expressed not as a hardcoded behavior, but as a request routed to a specifically named method on the object involved.

### From operator to method call

`a + b` is, mechanically, `type(a).__add__(a, b)`. You can call it exactly that way and get the identical result: 

In [ ]:
a = 3
b = 4
print(a.__add__(b))
print(type(a).__add__(a, b))

This isn't a curiosity or a trick — it's the literal mechanism. `+` is syntax; `__add__` is the actual method that syntax compiles down to invoking. Every operator you've used since Basics has a corresponding method:

```text
a + b       →  type(a).__add__(a, b)
a == b      →  type(a).__eq__(a, b)
len(a)      →  type(a).__len__(a)
a[i]        →  type(a).__getitem__(a, i)
for x in a  →  type(a).__iter__(a), then repeated type(iterator).__next__(iterator)
a(...)      →  type(a).__call__(a, ...)
```

Intermediate Python's Chapter 42 introduced dunder methods as a way to make *your own* classes participate in these operations. This chapter's point is the reverse direction: every one of Python's built-in types — `int`, `str`, `list`, `dict` — participates in exactly the same protocol you'd use, with no special privilege. There is no operator Python performs on a type's behalf that bypasses this lookup; the mechanism is uniform, all the way down to the built-in types themselves.

### Identity, equality, and the difference that matters

Two questions that sound almost interchangeable produce different answers depending on what's actually being asked: 

In [ ]:
a = [1, 2, 3]
b = [1, 2, 3]
c = a

print(a == b)
print(a is b)
print(a is c)

`==` asks `type(a).__eq__(a, b)` — a question you (or, for built-in types, Python) get to define the meaning of; for a list, it means "same elements, in the same order." `is` asks something the object can't override at all: *are these two names bound to the exact same object in memory* — the identity question, not a question of value. `a` and `b` hold equal but distinct list objects; `a` and `c` are the same object, reachable through two different names, exactly the aliasing relationship Basics' Chapter 4 first introduced with plain numbers.

`id()` exposes the underlying identity directly: 

In [ ]:
print(id(a))
print(id(b))
print(id(c))

`a` and `c` share an `id()`; `b` does not. This is worth being exact about because `is` is frequently misused where `==` was meant — comparing values with `is` can appear to work by coincidence for small integers or short strings (which CPython sometimes reuses as an implementation detail, not a language guarantee) and then fail unpredictably for larger or differently-constructed values. `is` is for identity questions specifically: checking against `None`, checking whether two names refer to the literal same object. Value comparison belongs to `==`.

### Mutability, revisited through the data model's lens

Basics distinguished mutable and immutable values informally — a list changes in place, a string doesn't. The data model gives this a precise mechanism: a type is mutable if it implements methods (like `__setitem__`, or `list.append`) that modify the object's internal state without creating a new object; it's immutable if no such method exists, and any apparent "change" — `x = x + 1` — actually rebinds the name to a freshly created object instead.

In [ ]:
numbers = [1, 2, 3]
print(id(numbers))
numbers.append(4)
print(id(numbers))    # same object -- mutated in place

n = 5
print(id(n))
n = n + 1
print(id(n))           # different object -- rebound, not mutated

### Protocols, not inheritance

None of the operations above required inheriting from a particular base class — a type participates in a protocol simply by implementing the relevant dunder methods, exactly the structural relationship Intermediate Python's Chapter 60 formalized with `Protocol`. This is the actual shift this chapter asks you to make: stop thinking of `len()`, `+`, indexing, and iteration as built-in language features with a fixed meaning, and start thinking of them as a small, fixed vocabulary of *questions* that Python asks any object it's handed — questions whose answers are supplied by whatever dunder methods that object's type happens to define. Everything from here through Chapter 65 — attribute lookup, descriptors, metaclasses — is really an extension of this same idea into progressively more fundamental territory: not just "how does an object respond to `+`," but "how does an object respond to being asked for an attribute at all," and eventually, "how does a *class itself* get built out of the syntax `class Foo:`."

### One problem

> Write a class `Vector2D` with `x` and `y` attributes that supports `+` (component-wise addition, returning a new `Vector2D`), `==` (component-wise equality), and `len()` returning `0` if both components are zero and `1` otherwise. Confirm `v1 + v2 == v3` works using only these three dunder methods.

In [ ]:
# TODO: define Vector2D with __add__, __eq__, and __len__


## 62. Attribute Lookup and Descriptors

Every one of these has worked without you asking how: 

In [ ]:
class Dog:
    species = "Canis familiaris"

    def __init__(self, name):
        self.name = name

    def bark(self):
        return f"{self.name} says woof"


rex = Dog("Rex")
print(rex.name)
print(rex.species)
print(rex.bark())

`rex.name`, `rex.species`, and `rex.bark` all use the same syntax — a dot followed by a name — yet they resolve completely differently underneath. `name` is an **instance attribute**, stored directly on `rex`. `species` is a **class attribute**, stored on `Dog`, shared by every instance (Intermediate Chapter 36). `bark` is a **method** — and this chapter's real subject is exactly what that word has been quietly hiding.

### Where each one actually lives

Every object carries its own attribute namespace, exposed through `__dict__`: 

In [ ]:
print(rex.__dict__)
print(Dog.__dict__.keys())

`rex.__dict__` holds only `name` — the one attribute actually set on this specific instance. `species` and `bark` live in `Dog.__dict__`, not `rex.__dict__` at all. When you write `rex.species`, Python doesn't find it on `rex` and doesn't stop there — it performs a **lookup**, checking a specific sequence of places until something matches.

```text
rex.species
     │
     ├── is "species" in rex.__dict__?         → no
     │
     └── is "species" in type(rex).__dict__?   → yes, found it
              (and, if not, continue up type(rex)'s MRO)
```

This is the same method-resolution-order machinery Intermediate Chapter 39 introduced for multiple inheritance, now framed as the general mechanism behind *every* attribute access, not just method calls in a diamond hierarchy.

### The part that isn't obvious: `bark` is not stored as a plain function on `rex`

Here's where it gets genuinely interesting. `Dog.__dict__["bark"]` is a plain function object — nothing about it, sitting in the class's namespace, knows anything about `self` yet.

In [ ]:
print(Dog.__dict__["bark"])
print(type(Dog.__dict__["bark"]))

And yet `rex.bark()` works, correctly supplying `rex` as `self`, without you ever writing `Dog.bark(rex)` yourself. Compare accessing it through the instance versus through the class directly: 

In [ ]:
print(rex.bark)
print(Dog.bark)

`rex.bark` is a **bound method** — `<bound method Dog.bark of <...Dog object...>>` — an object that already remembers `rex` and will supply it as `self` automatically. `Dog.bark` is the plain function itself, unbound to anything. Something transforms the plain function stored in `Dog.__dict__` into a bound method the moment it's accessed *through an instance* — and that something is a **descriptor**, the actual mechanism this entire chapter has been building toward.

### The descriptor protocol

A descriptor is any object that defines `__get__` (and, optionally, `__set__` or `__delete__`) and is itself stored as a class attribute. When attribute lookup finds a descriptor sitting in the class, it doesn't just hand it back — it calls the descriptor's `__get__`, and returns *that result* instead. Ordinary functions, it turns out, implement `__get__` themselves — which is the entire reason `bark` behaves the way it does.

In [ ]:
def plain_function():
    pass

print(hasattr(plain_function, "__get__"))

Every function object is already a descriptor. When Python looks up `bark` on `Dog` and finds a function there, and that lookup happened *through an instance* (`rex.bark`, not `Dog.bark`), it calls the function's `__get__(rex, Dog)`, which returns a bound method wrapping `rex` and the original function together. This is not a special case Python hardcodes for methods — it's the descriptor protocol operating exactly as it would for any other descriptor, and methods simply happen to be the first, most common example of one.

### Data descriptors versus non-data descriptors — and why the distinction changes lookup order

A descriptor that defines only `__get__` is a **non-data descriptor**. One that also defines `__set__` (or `__delete__`) is a **data descriptor**. This distinction isn't a naming curiosity — it changes the actual priority order lookup follows.

```text
attribute lookup order:
1. data descriptors found on the class (or its MRO)
2. instance __dict__
3. non-data descriptors and plain class attributes
4. __getattr__, if defined (next chapter)
```

A data descriptor takes priority over an instance's own `__dict__`; a non-data descriptor — which is exactly what an ordinary function is — does not. This is precisely why an instance *can* shadow a method by assigning an attribute of the same name directly onto itself, while it *cannot* similarly override a property (a data descriptor, covered in the next two chapters) the same way.

In [ ]:
class Example:
    def greet(self):
        return "hello from the method"


e = Example()
print(e.greet())

e.greet = lambda: "hello from the instance"
print(e.greet())

Assigning `e.greet = lambda: ...` created an entry directly in `e.__dict__`, and because `Example.greet` is only a non-data descriptor (a plain function), the instance's own `__dict__` entry wins the lookup — the class's method is shadowed entirely for `e`. Chapter 64 will show a data descriptor deliberately built to *prevent* exactly this kind of shadowing, which is often precisely the point of writing one.

### One problem

> Given a class with both a class attribute and, on one specific instance, an instance attribute of the same name, write code that prints `__dict__` for both the instance and the class to show exactly where each value actually lives, and explain — in a comment — which one attribute lookup would find and why.

In [ ]:
# TODO: demonstrate class-attribute vs instance-attribute shadowing via __dict__


## 63. `__getattribute__`, `__getattr__`, and Dynamic Attributes

The last chapter described attribute lookup as a fixed sequence: instance `__dict__`, then class (and its MRO), respecting data-descriptor priority along the way. That entire sequence is itself just the *default implementation* of one specific method — `__getattribute__` — and, like everything else in the data model, it can be overridden.

### `__getattribute__` — intercepting every single attribute access

`__getattribute__` is called for **every** attribute access on an object, without exception — `obj.anything` always goes through it first, before any of the lookup rules from the last chapter even apply, because those rules are what the default `__getattribute__` implements.

In [ ]:
class Noisy:
    def __init__(self, value):
        self.value = value

    def __getattribute__(self, name):
        print(f"looking up: {name}")
        return object.__getattribute__(self, name)


n = Noisy(42)
print(n.value)

Notice `object.__getattribute__(self, name)` inside the override — this calls the *default* implementation, the ordinary lookup sequence from the last chapter, to actually retrieve the value once the interception has done its logging. This is not optional decoration; it's the only correct way to actually finish the lookup once you've overridden `__getattribute__` — write the method without eventually delegating to `object.__getattribute__` (or a `super()` call reaching it), and normal attribute access breaks entirely.

### The infinite recursion trap

This is worth showing directly, because it's the single most common mistake when overriding attribute access, and it's genuinely easy to write by accident.

In [ ]:
class Broken:
    def __init__(self, value):
        self.value = value

    def __getattribute__(self, name):
        return self.__dict__[name]   # accessing self.__dict__ triggers __getattribute__ again


b = Broken(42)
b.value

`RecursionError`. `self.__dict__` is itself an attribute access — it goes through `__getattribute__` too, which calls `self.__dict__` again to serve *that* request, which calls `__getattribute__` again, without end. This is exactly why the correct version above reaches for `object.__getattribute__(self, name)` specifically — a call that bypasses `Noisy`'s own override and goes straight to the base implementation, breaking the cycle deliberately.

### `__getattr__` — a much safer, much more common tool

`__getattribute__` intercepts *every* access, which makes it powerful and easy to break. `__getattr__` is a gentler alternative: it's only called as a **fallback**, when normal lookup — the entire sequence from the last chapter — has already failed to find the attribute anywhere.

```text
obj.name
    ↓
__getattribute__ runs the normal lookup sequence
    ↓
found it?  →  yes: return it, done. __getattr__ is never even called.
    ↓  no
__getattr__(obj, "name") is called, as a last resort
```

In [ ]:
class Config:
    def __init__(self):
        self.debug = True

    def __getattr__(self, name):
        return f"no setting named {name!r}"


c = Config()
print(c.debug)          # found normally -- __getattr__ never runs
print(c.timeout)        # not found normally -- __getattr__ provides a fallback

`c.debug` resolves through the ordinary instance `__dict__` lookup, exactly as always — `__getattr__` is completely uninvolved. `c.timeout` doesn't exist anywhere in the normal lookup chain, so Python falls back to calling `__getattr__(c, "timeout")`, and whatever that returns becomes the result. This is a far safer place to add custom behavior than `__getattribute__`, precisely because it can't interfere with attributes that already resolve normally — there's no risk of accidentally breaking `self.value` the way the `Broken` example did, since `__getattr__` for `value` would simply never be reached if `value` already exists.

### A genuinely practical use: dynamic, computed attributes

`__getattr__` is the mechanism behind objects that expose data lazily or dynamically — a configuration object backed by a dictionary, for instance, where every key should be accessible as an attribute without writing one property per key.

In [ ]:
class DynamicRecord:
    def __init__(self, data):
        self._data = data

    def __getattr__(self, name):
        try:
            return self._data[name]
        except KeyError:
            raise AttributeError(f"{type(self).__name__!r} has no attribute {name!r}")


record = DynamicRecord({"name": "Ali", "age": 25})
print(record.name)
print(record.age)
record.city

Two details here matter beyond the obvious convenience. First, `self._data` itself is found through ordinary lookup (it's set directly in `__init__`), so referencing it inside `__getattr__` does not trigger `__getattr__` again — there's no recursion trap here, unlike the `__getattribute__` example, because `__getattr__` is only ever consulted when normal lookup has *already failed*, and `_data` never fails normal lookup.

Second, notice the deliberate `raise AttributeError(...)` for a genuinely missing key, rather than returning `None` or some other placeholder silently. This matters more than it looks: `hasattr()`, and a great deal of other Python machinery, work by calling `getattr()` and checking whether an `AttributeError` was raised. A `__getattr__` that swallows a missing name and returns something else instead breaks that contract silently.

In [ ]:
print(hasattr(record, "name"))
print(hasattr(record, "salary"))

### The danger worth stating plainly

Overriding attribute access — either method — makes an object's behavior harder to predict by reading its class definition alone, because `obj.anything` might no longer mean "look this up in the usual places." Used deliberately, for a specific, well-documented purpose (dynamic configuration, proxying to another object, lazy-loaded data), this is a legitimate and common tool. Used casually, it makes debugging significantly harder, because a plain-looking `.` in code stops being a reliable signal of plain, predictable behavior.

### One problem

> Write a class `ReadOnlyProxy` that wraps another object in `__init__`, and whose `__getattr__` forwards any attribute lookup that isn't found directly on the proxy to the wrapped object — but whose `__setattr__` (research this one briefly) raises an `AttributeError` for any attempt to set a new attribute, effectively making the wrapped object's data readable but never writable through the proxy.

In [ ]:
# TODO: define ReadOnlyProxy using __getattr__ and __setattr__


## 64. Descriptors in Practice

Chapter 62 established that ordinary functions are descriptors, and that this is what makes methods bind correctly. This chapter builds one deliberately, from the same protocol, and then shows that a tool from Intermediate Python — `@property` — was one all along.

### The full protocol

```text
__get__(self, instance, owner)     called when the attribute is read
__set__(self, instance, value)     called when the attribute is assigned to
__delete__(self, instance)         called when the attribute is deleted with del
```

A descriptor implementing only `__get__` is non-data (Chapter 62); implementing `__set__` or `__delete__` as well makes it a data descriptor, which — as established — takes priority over an instance's own `__dict__`.

### Building a validating descriptor

Here's the problem Intermediate Chapter 43 solved with a single property, generalized: suppose several different attributes across several different classes all need the same "must be a positive number" validation, and rewriting that validation as a separate property in every single class would duplicate the same handful of lines repeatedly.

In [ ]:
class PositiveNumber:
    def __set_name__(self, owner, name):
        self.name = "_" + name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return getattr(instance, self.name)

    def __set__(self, instance, value):
        if value <= 0:
            raise ValueError(f"{self.name[1:]} must be positive, got {value}")
        setattr(instance, self.name, value)


class Rectangle:
    width = PositiveNumber()
    height = PositiveNumber()

    def __init__(self, width, height):
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height


r = Rectangle(4, 5)
print(r.area())

r.width = -3

`PositiveNumber` is written **once**, and `width` and `height` on `Rectangle` — and any other class, for any other attribute — reuse it directly, without duplicating the validation logic anywhere. This is the actual payoff descriptors offer over a hand-written `@property` per attribute: the validation behavior is factored out into its own reusable, independent object, rather than copy-pasted (with slightly different names) into every class and attribute that happens to need the same rule.

### Tracing what actually happens on `r.width = 4`

This is worth walking through explicitly, because the mechanism is genuinely a few layers deep. `Rectangle.width` is set to a `PositiveNumber()` instance — a **class** attribute, shared by every `Rectangle`. `r.width = 4` does not create an instance attribute called `width` directly; because `PositiveNumber` is a data descriptor (it defines `__set__`), Chapter 62's priority rule applies: the descriptor intercepts the assignment before it would ever reach `r.__dict__["width"]`. Python instead calls `Rectangle.width.__set__(r, 4)`, which validates `4`, and — inside that method — stores the *actual* value under a different name, `r._width`, using ordinary `setattr`.

```text
r.width = 4
     ↓
Rectangle.width is a data descriptor  →  intercepted, not stored directly on r
     ↓
Rectangle.width.__set__(r, 4) runs
     ↓
validates, then stores under r._width instead
```

Reading `r.width` later follows the same interception in reverse: `PositiveNumber.__get__` runs, and fetches `r._width`. The name `width` on the instance is never actually populated at all — every read and write is routed, invisibly, through the descriptor.

### `__set_name__` — how the descriptor knows its own name

`__set_name__(self, owner, name)` is called automatically by Python, once, at class-creation time — specifically when the class body finishes executing — telling each descriptor what name it was assigned to (`width`, `height`). This is what lets one single `PositiveNumber` class correctly store `width` under `_width` and `height` under `_height`, without the author of `PositiveNumber` needing to hardcode either name, or the user of `PositiveNumber` needing to pass the name in manually.

### `@property` is a descriptor

This is worth confirming directly rather than taking on faith: 

In [ ]:
print(hasattr(property, "__get__"))
print(hasattr(property, "__set__"))

`property` is a built-in class that implements exactly the descriptor protocol — `@property` and its `@x.setter` companion from Intermediate Chapter 43 were never separate language magic; they were a convenient, ready-made way of constructing one specific, common kind of descriptor (one tied to a single pair of getter/setter functions) without writing the `__get__`/`__set__` machinery by hand each time. `PositiveNumber`, above, is what you reach for once the same validation needs to apply across many attributes or many classes, rather than once per property.

### One problem

> Write a descriptor `Typed(expected_type)` that enforces a specific type on assignment (raising `TypeError` if the assigned value isn't an instance of `expected_type`), using `__set_name__` the same way `PositiveNumber` did. Use it for at least two differently-typed attributes on one class.

In [ ]:
# TODO: define Typed as a reusable, type-checking data descriptor


## 65. Metaclasses — Classes That Create Classes

Every chapter so far in this part has taken the existence of `class Foo:` for granted and looked at what happens *inside* the resulting object. This chapter asks a different question: what actually creates `Foo` itself, as an object, when the `class` statement runs?

### Classes are objects too

This sounds like a throwaway remark until you actually check it: 

In [ ]:
class Dog:
    pass


print(type(Dog))
print(isinstance(Dog, object))

`Dog` — the class itself, not an instance of it — has a type, and that type is `type`. `Dog` is an *instance* of `type`, in exactly the same sense that `rex = Dog()` would make `rex` an instance of `Dog`. This gives Python a precise three-level relationship, worth stating exactly:

```text
object       an ordinary instance, e.g. rex
   ↑ instance of
class        e.g. Dog -- itself an object
   ↑ instance of
metaclass    e.g. type -- the "class of a class"
```

`type` is the default **metaclass** — the class whose instances are themselves classes. Every class you have ever written, across all three volumes of this series, is an instance of `type`, whether you ever noticed or not.

### `type` as a function that builds classes

`type` can be called directly, with three arguments, and doing so builds a class exactly as the `class` statement would — because the `class` statement is, underneath, translated into precisely this call.

In [ ]:
def bark(self):
    return f"{self.name} says woof"


Dog = type("Dog", (), {"species": "Canis familiaris", "bark": bark, "__init__": lambda self, name: setattr(self, "name", name)})

rex = Dog("Rex")
print(rex.bark())
print(rex.species)

`type(name, bases, namespace)`: `name` is the class's name as a string; `bases` is a tuple of base classes (empty here, meaning it inherits only from `object`); `namespace` is a dictionary of everything that would otherwise sit in the class body — methods, class attributes. This `Dog` is functionally identical to one written with ordinary `class` syntax; `type(...)` and `class Foo:` are two different spellings of the exact same underlying construction step.

### What a metaclass is

If classes are created by calling `type(...)`, then a **metaclass** is simply a *different* callable used in `type`'s place — a class that inherits from `type` itself, and can therefore customize what happens when a new class is built from it.

In [ ]:
class LoudMeta(type):
    def __new__(mcs, name, bases, namespace):
        print(f"Creating class: {name}")
        return super().__new__(mcs, name, bases, namespace)


class Dog(metaclass=LoudMeta):
    def bark(self):
        return "woof"


rex = Dog()
print(rex.bark())

`"Creating class: Dog"` printed the moment the `class Dog(metaclass=LoudMeta):` statement itself executed — not when `Dog()` was later called to make an instance. `metaclass=LoudMeta` tells Python: instead of building `Dog` by calling `type(name, bases, namespace)`, build it by calling `LoudMeta(name, bases, namespace)` — and because `LoudMeta` inherits from `type`, it can override `__new__` to intercept, inspect, or modify the class *as it's being constructed*, before it's ever used to make a single instance.

### `__new__` versus `__init__`, one level up

This mirrors a distinction ordinary classes already have (`__new__` creates the object, `__init__` sets it up), just applied to class creation instead of instance creation. `LoudMeta.__new__` here calls `super().__new__(mcs, name, bases, namespace)`, delegating the actual class-building work back to `type`, after running its own logic first — exactly the pattern Intermediate Chapter 38's `super()` established for ordinary inheritance, now one level higher up the object model.

### What metaclasses are actually good for — stated honestly

It's worth being direct about something the curriculum for this chapter specifically asked not to obscure: **most Python code never needs a metaclass.** The overwhelming majority of problems that look like they need one are better solved with a decorator (which can also inspect and modify a class after it's built, more simply — Intermediate Chapter 48 already gave you the tools for this), an `__init_subclass__` hook (a lighter-weight alternative for a common subset of metaclass use cases), or ordinary inheritance.

Where metaclasses genuinely earn their place is in framework code that needs to do something to *every* class of a certain kind, automatically, at the moment it's defined — before any instance ever exists. A well-known real example: Python's own `ABCMeta` (used implicitly whenever you inherit from `ABC`, as Intermediate Chapter 41 taught) is a metaclass, and it's precisely what makes `Shape()` raise `TypeError` the instant you try to instantiate a class with unimplemented abstract methods — that check has to happen during class construction and instantiation, at a point ordinary inheritance and decorators don't naturally reach.

### The relationship, stated once, plainly

`object → class → metaclass` is not an escalating hierarchy of "more advanced" features stacked for their own sake. It's the same single mechanism — an object built by calling something, using a namespace of attributes — applied recursively one level further than most code ever needs to look. Understanding that it's there, and roughly why, matters far more than being fluent in writing one; the honest, professional instinct, reinforced repeatedly in real Python codebases, is to reach for a metaclass only after confirming that a decorator or `__init_subclass__` genuinely can't do the job.

### One problem

> Write a metaclass `SingletonMeta` that ensures any class using it as its metaclass can only ever have one instance — calling the class a second time returns the same object created the first time, rather than constructing a new one. Demonstrate it with a small `Configuration` class.

In [ ]:
# TODO: define SingletonMeta and a Configuration class using it


# Part II — Functional and Advanced Language Features

## 66. Advanced Closures and Scope

Intermediate Chapter 47 established what a closure is: a nested function that captures a variable from its enclosing scope, and `nonlocal` as the tool for actually reassigning that captured variable rather than shadowing it. This chapter goes one level deeper into exactly what "capture" means, because the informal picture — "the function remembers the value" — is subtly wrong, and the specific way it's wrong causes a genuinely famous class of bugs.

### LEGB, stated precisely

Every name lookup in Python is resolved by checking a fixed sequence of scopes, commonly abbreviated **LEGB**:

```text
Local       the innermost function currently executing
Enclosing   any enclosing function's scope (for a nested function)
Global      the module's top-level scope
Built-in    Python's built-in names (len, print, range, ...)
```

A name is resolved by checking each of these, in this exact order, stopping at the first match.

In [ ]:
x = "global"


def outer():
    x = "enclosing"

    def inner():
        x = "local"
        print(x)

    inner()
    print(x)


outer()
print(x)

Each `print(x)` finds a different `x`, because each one runs in a different scope, and LEGB always starts its search at the *innermost* scope of wherever the code physically is — never at the "most recently touched" scope, and never anywhere else.

### What "capturing" actually means — a variable, not a value

Here is the behavior that separates a correct mental model of closures from an almost-correct one that will eventually mislead you.

In [ ]:
def make_functions():
    functions = []
    for i in range(3):
        def show():
            print(i)
        functions.append(show)
    return functions


fs = make_functions()
for f in fs:
    f()

What would you expect this to print, before running it — `0, 1, 2`, or something else?

Run it. It prints `2, 2, 2`. Every single closure in `functions` shares the *same* captured `i`, not a snapshot of whatever `i` happened to be at the moment each `show` was defined. This is **late binding**: a closure captures a *reference to the variable itself* — the enclosing scope's actual storage location for `i` — not the value that variable held at definition time. By the time any of the three `show` functions actually runs, the loop has already finished, and `i`'s final value, `2`, is whatever all three closures see, because all three were always looking at the same `i`, never a copy of it.

```text
functions[0], functions[1], functions[2]
        │            │            │
        └────────────┴────────────┘
                     all point to
                the same captured 'i'
```

This is exactly the kind of surprising, genuinely common bug the informal "closures remember values" model hides from you — and it's worth internalizing as *late binding*, specifically, rather than a vague "loops and closures don't mix," because the fix follows directly from understanding the actual mechanism rather than avoiding the pattern entirely.

### The fix: force a new binding per iteration

If each closure needs its *own* independent value rather than a shared reference to the loop variable, that value needs to be captured through something Python resolves eagerly — a default argument value, evaluated once, at function-definition time, rather than looked up later at call time.

In [ ]:
def make_functions():
    functions = []
    for i in range(3):
        def show(i=i):   # default value captured NOW, at definition time
            print(i)
        functions.append(show)
    return functions


fs = make_functions()
for f in fs:
    f()

`def show(i=i):` looks unusual the first time you see it, but the mechanism is exactly Basics' default-parameter-value rule (Chapter 23): default values are evaluated once, immediately, when the `def` statement runs — not later, when the function is called. `i=i` deliberately exploits this to freeze the *current* loop iteration's value of `i` into a genuinely separate default argument for each `show`, rather than leaving `i` to be looked up fresh, later, through the shared enclosing scope.

### Closure factories, and why the pattern generalizes

The "function that builds and returns other functions" shape from Intermediate Chapter 47 (`make_multiplier`, `make_counter`) is worth naming explicitly as a **closure factory** — a function whose entire purpose is producing configured, independent closures. The late-binding trap above only bites when a *loop* builds several closures that all reference the *same* enclosing variable; a factory function like `make_counter()`, called separately each time, doesn't have this problem, because each call genuinely creates a brand-new, independent enclosing scope with its own separate variable — there's no shared loop variable for multiple closures to accidentally alias.

### Debugging a closure: what to actually look at

When a closure behaves unexpectedly, `__closure__` (briefly shown in Intermediate Chapter 47) tells you exactly what's captured, right now, rather than requiring you to guess. Going back to the broken version, before the default-argument fix: 

In [ ]:
def make_functions_broken():
    functions = []
    for i in range(3):
        def show():
            print(i)
        functions.append(show)
    return functions


fs = make_functions_broken()
for f in fs:
    print(f.__closure__[0].cell_contents)

All three report `2` — direct, inspectable confirmation that every closure's cell points at the exact same captured `i`, exactly as the surprising output predicted. Now check the *fixed* version the same way: 

In [ ]:
print(fs[0].__closure__)

fixed_fs = make_functions()
print(fixed_fs[0].__closure__)
print(fixed_fs[0].__defaults__)

The fixed version's `__closure__` is `None` — there's nothing to inspect there at all, because `i=i` never created a free variable captured from an enclosing scope in the first place; it created an ordinary default *argument*, and default arguments live in `__defaults__`, exactly the same mechanism Basics' Chapter 23 introduced for `greet(name, greeting="Hello")`. This is worth stating precisely, because it corrects an easy-to-make assumption: the fix didn't give each closure its own private *cell* — it sidestepped closures entirely for that value, by capturing it as a default argument instead, which is exactly why `__closure__` and `__defaults__` are the two places worth checking, in that order, whenever a closure's behavior needs verifying rather than guessing.

### One problem

> Write a function `make_validators(rules)` where `rules` is a dictionary mapping field names to minimum values, that returns a dictionary mapping each field name to a validator function `lambda value: value >= minimum` — using the default-argument trick to avoid the late-binding trap, and demonstrate that each returned validator enforces its own field's minimum correctly.

In [ ]:
# TODO: define make_validators avoiding the late-binding trap


## 67. Advanced Decorators

Intermediate Chapter 48 built decorators from first principles: a closure wraps a function, `@decorator` is syntax for reassigning a name to the wrapped result, and `functools.wraps` repairs the wrapped function's lost identity. This chapter extends that foundation into shapes that show up constantly in real frameworks and libraries.

### Recalling the shape, briefly

```text
def decorator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        # before
        result = func(*args, **kwargs)
        # after
        return result
    return wrapper
```

Everything in this chapter is a variation on exactly this shape — different numbers of layers, different targets (methods, whole classes), different degrees of state carried between calls.

### Decorators on methods — `self` doesn't disappear, it just becomes the first positional argument

A decorator written for a plain function works on a method with no changes at all, because `*args` simply captures `self` along with everything else.

In [ ]:
from functools import wraps
import time


def log_calls(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"calling {func.__name__}")
        return func(*args, **kwargs)
    return wrapper


class Account:
    def __init__(self, balance):
        self.balance = balance

    @log_calls
    def withdraw(self, amount):
        self.balance -= amount
        return self.balance


a = Account(100)
print(a.withdraw(30))

`wrapper(*args, **kwargs)` receives `a` as `args[0]` (Python's ordinary method-binding mechanism from Chapter 62 still applies — `wrapper` itself is a function, hence a descriptor, hence it binds correctly), followed by `30`. Nothing about the decorator needed to know it was wrapping a method rather than a plain function; `*args, **kwargs` makes the wrapper argument-shape-agnostic, which is exactly why that signature, rather than a fixed one, is close to universal in real decorator code.

### Class decorators — modifying a class instead of a function

A decorator can be applied to a class, not just a function — and since a class is just an object (Chapter 65), a class decorator is simply a function that receives a class and returns something, exactly the same shape as a function decorator, one level up.

In [ ]:
def add_repr(cls):
    def __repr__(self):
        fields = ", ".join(f"{k}={v!r}" for k, v in self.__dict__.items())
        return f"{cls.__name__}({fields})"
    cls.__repr__ = __repr__
    return cls


@add_repr
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y


p = Point(3, 4)
print(p)

`add_repr` doesn't wrap `Point` in anything — it takes the class, attaches a new `__repr__` method directly onto it, and returns the *same* class object, modified in place. This is a lighter-weight alternative to a metaclass for exactly the situations Chapter 65 pointed toward: doing something to a class automatically, without needing the full class-construction interception a metaclass provides.

### Stateful decorators — remembering something across calls

A decorator's `wrapper` closure can hold state that persists between separate calls to the decorated function, using exactly the closure mechanism Chapter 66 just examined in detail.

In [ ]:
def count_calls(func):
    call_count = 0

    @wraps(func)
    def wrapper(*args, **kwargs):
        nonlocal call_count
        call_count += 1
        print(f"{func.__name__} has been called {call_count} time(s)")
        return func(*args, **kwargs)

    return wrapper


@count_calls
def greet(name):
    return f"Hello, {name}"


print(greet("Ali"))
print(greet("Sara"))
print(greet("Bilal"))

`call_count` is declared once, in `count_calls`'s scope, and `nonlocal call_count` inside `wrapper` — exactly the tool from Chapter 66 and Intermediate Chapter 47 — is what allows repeated calls to `greet` to actually accumulate a running total, rather than each call seeing a freshly reset value.

### Stacked decorators, layered carefully

Recall from Intermediate Chapter 48 that stacked decorators apply bottom-up. Combining a decorator factory (one that takes arguments) with a stateful one makes this concrete in a slightly more elaborate case.

In [ ]:
def retry(times):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            last_exception = None
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except ValueError as e:
                    last_exception = e
                    print(f"attempt {attempt} failed: {e}")
            raise last_exception
        return wrapper
    return decorator


attempt_counter = {"count": 0}


@retry(times=3)
def flaky_operation():
    attempt_counter["count"] += 1
    if attempt_counter["count"] < 3:
        raise ValueError("not ready yet")
    return "success"


print(flaky_operation())

`retry(times=3)` first produces `decorator`, which is then applied to `flaky_operation`, exactly matching the two-level structure Intermediate Chapter 48 introduced with `repeat`. The genuinely new piece here is that `wrapper` doesn't just call `func` once — it actively controls the calling behavior, retrying on a specific, anticipated failure, which is the kind of thing decorators are particularly well suited for: behavior that wraps *around* a call, potentially calling the wrapped function more than once, or not at all, based on what happens.

### Preserving behavior carefully

A decorator that changes what a function actually computes, rather than merely observing or retrying it, needs to be introduced with real caution — a decorator's whole value depends on the wrapped function still being usable and predictable to whoever calls it. A decorator that silently swallows exceptions, alters return values in surprising ways, or has side effects unrelated to its stated purpose makes code harder to reason about specifically because the decoration is invisible at the call site — `flaky_operation()` gives no visual hint that retries are happening underneath, which is a genuine cost worth weighing against the convenience, especially in code other people will read without immediately seeing the `@retry(times=3)` line above the definition.

### One problem

> Write a decorator `deprecated(replacement)` that, when applied to a function, prints a warning the first time (and only the first time) that function is called, telling the caller to use `replacement` instead, then calls the original function normally on every call, including the first.

In [ ]:
from functools import wraps

# TODO: define deprecated(replacement) as a decorator factory


## 68. Function Signatures and `inspect`

A function object carries far more information about itself than its name and behavior when called. This matters because a large amount of real, professional Python — testing frameworks, dependency-injection libraries, command-line tools, IDEs offering autocomplete — works by examining functions *before* calling them, rather than only calling them.

In [ ]:
def greet(name: str, greeting: str = "Hello", *, shout: bool = False) -> str:
    result = f"{greeting}, {name}"
    return result.upper() if shout else result


print(greet.__name__)
print(greet.__defaults__)
print(greet.__annotations__)

`__name__`, `__defaults__`, and `__annotations__` are attributes sitting directly on the function object, readable without calling it at all. `inspect`, the standard-library module built specifically around this kind of introspection, provides a considerably richer, more structured view than reading these raw attributes by hand.

### `inspect.signature` — a structured view of a callable's parameters

```python
inspect.signature(func)
```

returns a `Signature` object describing exactly what `func` expects, structured enough to actually reason about programmatically, rather than parsing `__defaults__` and `__annotations__` separately and trying to line them up yourself.

In [ ]:
import inspect

sig = inspect.signature(greet)
print(sig)

for name, param in sig.parameters.items():
    print(name, "-", "default:", param.default, "| annotation:", param.annotation, "| kind:", param.kind)

Each `Parameter` reports its default value (`inspect.Parameter.empty` if there isn't one), its annotation, and its **kind** — whether it's positional-or-keyword, keyword-only (`shout`, here, appears after the bare `*` in the definition), and so on. This is precisely the structured information a tool would need to, for instance, generate a command-line interface automatically from a function's signature, or validate that a set of arguments matches what a function actually accepts, before ever calling it.

### Binding arguments without calling the function

`Signature.bind` performs the same argument-matching Python does internally when a function is actually called — matching positional and keyword arguments against parameter names, applying defaults — but as a separate, inspectable step.

In [ ]:
bound = sig.bind("Ali", shout=True)
bound.apply_defaults()

print(bound.arguments)

`bound.arguments` shows every parameter mapped to its actual value — including `greeting`, filled in from its default via `apply_defaults()`, even though the original call never mentioned it. This lets code inspect exactly what a call *would* look like, fully resolved, without triggering any of the function's actual behavior.

### Why frameworks lean on this so heavily

Consider, concretely, what a testing framework's fixture system (Intermediate Chapter 62's `pytest` fixtures) has to do: given a test function, it needs to figure out which fixtures that specific function is asking for, by name, and supply exactly those — without the test author writing any manual registration code. That's only possible because `pytest` can call `inspect.signature` on the test function, read off its parameter names, and match them against known fixtures, entirely before ever calling the test itself. Dependency-injection frameworks, ORMs that validate model definitions, and IDEs offering autocomplete all lean on some version of the same capability: treating a function's signature as data to be examined, not just a contract to be silently honored at call time.

### `getsource` and its limits

`inspect` can also retrieve a function's actual source code, when it's available: 

In [ ]:
print(inspect.getsource(greet))

This depends on the function's source actually being retrievable from a real file on disk — it fails for functions defined interactively in some contexts, or built dynamically (recall Chapter 65's `type(...)` construction, which has no source file to point to at all). It's a genuinely useful tool for building developer-facing tooling — documentation generators, debuggers — but it's worth knowing its limits rather than assuming it always works.

### One problem

> Write a function `describe_function(func)` that uses `inspect.signature` to print, for any given function, each parameter's name, whether it has a default, and what that default is (or `"required"` if not), formatted as a readable summary line per parameter.

In [ ]:
import inspect

# TODO: define describe_function(func) using inspect.signature


## 69. Higher-Order Programming with `functools`

Intermediate Chapter 49 introduced `wraps`, `lru_cache`, and `partial`. This chapter extends the same "functions as first-class values" theme with tools for two related but different problems: pre-configuring methods specifically, reducing a sequence to a single value, and — the chapter's most substantial new idea — writing one function that behaves differently depending on the *type* of its argument, without a chain of `isinstance` checks.

### `partialmethod` — `partial`, specifically for methods

`functools.partial` (Intermediate Chapter 49) pre-fills arguments for an ordinary function. `partialmethod` does the equivalent inside a class body, correctly accounting for the fact that a method's first argument, `self`, isn't known until an instance actually exists.

In [ ]:
from functools import partialmethod


class TextProcessor:
    def __init__(self, text):
        self.text = text

    def replace(self, old, new):
        self.text = self.text.replace(old, new)
        return self.text

    remove_spaces = partialmethod(replace, " ", "")
    remove_dashes = partialmethod(replace, "-", "")


t = TextProcessor("hello - world")
print(t.remove_dashes())
print(t.remove_spaces())

`partial` itself doesn't work correctly here — it would try to bind `self`'s position too early, before any instance exists. `partialmethod` specifically defers that binding until the method is actually accessed through an instance, correctly leaving `self` to be supplied normally, and pre-filling only the arguments *after* it.

### `reduce` — collapsing a sequence into one value

Basics and Intermediate both used `sum()` freely without asking how a general "combine everything into one value" operation might be written by hand. `functools.reduce` is that general tool.

In [ ]:
from functools import reduce

numbers = [1, 2, 3, 4, 5]

total = reduce(lambda acc, n: acc + n, numbers)
print(total)

product = reduce(lambda acc, n: acc * n, numbers)
print(product)

`reduce(function, iterable)` repeatedly calls `function(accumulated_value, next_item)`, carrying the result forward each time, until the sequence is exhausted.

```text
reduce(f, [1, 2, 3, 4, 5])
    f(1, 2)       = 3
    f(3, 3)       = 6
    f(6, 4)       = 10
    f(10, 5)      = 15
```

`sum()` is, conceptually, `reduce` specialized to addition with a starting value of `0` — worth knowing as a mental model, though `sum()` remains the right tool whenever addition specifically is all you need, precisely because it's more readable than the equivalent `reduce` call. `reduce` earns its place for combining operations that don't already have a dedicated built-in — folding a list of dictionaries into one merged dictionary, for instance — and it's worth being honest that a plain loop is very often *more* readable than `reduce` for exactly this kind of case; reaching for `reduce` should follow from it genuinely clarifying the code, not from a general instinct to avoid loops.

### `cache` versus `lru_cache`

Python 3.9 added `functools.cache`, which is `lru_cache` with no size limit at all — a simpler spelling for the common case where you want every past call's result remembered permanently, rather than only the most recent ones.

In [ ]:
from functools import cache


@cache
def fibonacci(n):
    if n < 2:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)


print(fibonacci(30))

### `singledispatch` — one function name, behavior chosen by argument type

Here is a genuinely common shape of problem: a function needs to behave differently depending on what type of argument it receives, and the obvious approach accumulates `isinstance` checks that get harder to extend cleanly over time.

In [ ]:
def describe(value):
    if isinstance(value, int):
        return f"an integer: {value}"
    elif isinstance(value, str):
        return f"a string of length {len(value)}"
    elif isinstance(value, list):
        return f"a list with {len(value)} items"
    else:
        return f"something else: {value!r}"


print(describe(42))
print(describe("hello"))
print(describe([1, 2, 3]))

This works, but every new type this function needs to support means editing this one growing function, and the logic for handling `list` sits physically next to the logic for handling `str`, with no natural separation. `functools.singledispatch` restructures this: one function is registered per type, and Python chooses which one runs based on the type of the first argument.

In [ ]:
from functools import singledispatch


@singledispatch
def describe(value):
    return f"something else: {value!r}"


@describe.register
def _(value: int):
    return f"an integer: {value}"


@describe.register
def _(value: str):
    return f"a string of length {len(value)}"


@describe.register
def _(value: list):
    return f"a list with {len(value)} items"


print(describe(42))
print(describe("hello"))
print(describe([1, 2, 3]))
print(describe(3.14))

`@singledispatch` marks `describe` as the default implementation, used whenever no more specific registration matches. Each `@describe.register` — using the type annotation on its single parameter to declare which type it handles — adds a new implementation, without touching the original function's body at all. `describe(3.14)` falls through to the default, since no registration was made for `float`. This is the practical benefit over the `isinstance` chain: adding support for a new type never requires editing existing code, only adding a new, independent registration — a real instance of the general software-design principle "open for extension, closed for modification," made concrete through a specific standard-library tool rather than left as an abstract slogan.

### One problem

> Using `singledispatch`, write a function `to_json_value(value)` that converts a Python value into a JSON-compatible representation: `int`/`float`/`str`/`bool` pass through unchanged, `list` recursively converts each element, and `dict` recursively converts each value — with a default case raising `TypeError` for anything else.

In [ ]:
from functools import singledispatch

# TODO: define to_json_value using singledispatch and .register


## 70. Pattern Matching with `match` / `case`

Python 3.10 added structural pattern matching — syntax that looks superficially like a `switch` statement from other languages, but does something considerably more capable: it can inspect the *shape* of a value, not just compare it against a single literal.

### The simplest case — matching against literals

At its most basic, `match`/`case` genuinely does resemble a cleaner `if`/`elif` chain: 

In [ ]:
def describe_status(code):
    match code:
        case 200:
            return "OK"
        case 404:
            return "Not Found"
        case 500:
            return "Server Error"
        case _:
            return "Unknown status"


print(describe_status(200))
print(describe_status(404))
print(describe_status(999))

`case _:` is the **wildcard pattern** — it matches anything, and by convention plays the role `else` plays in an `if` chain, catching whatever didn't match an earlier, more specific case. Written only this way, `match` is a slightly more readable alternative to a long `if`/`elif` chain of equality checks, but nothing more — the real capability starts with the next kind of pattern.

### Sequence patterns — matching, and unpacking, a shape at once

Intermediate Chapter 32 taught unpacking a fixed-length sequence into names. `match` lets you combine "does this have the right shape" and "unpack it into names" into a single step, with different cases for different shapes.

In [ ]:
def handle_command(command):
    match command:
        case ["quit"]:
            return "Exiting"
        case ["move", direction]:
            return f"Moving {direction}"
        case ["move", direction, distance]:
            return f"Moving {direction} by {distance}"
        case _:
            return "Unrecognized command"


print(handle_command(["quit"]))
print(handle_command(["move", "north"]))
print(handle_command(["move", "north", "10"]))
print(handle_command(["dance"]))

Each `case` here checks *both* the length of the list and, simultaneously, binds `direction` (and, in the third case, `distance`) to whatever values are actually present — there's no separate `len(command) == 2` check followed by a separate unpacking step; the pattern expresses both requirements at once, and Python only proceeds into that branch if the shape genuinely matches.

`*rest`-style patterns work here too, exactly mirroring Chapter 32's unpacking syntax: 

In [ ]:
def summarize(items):
    match items:
        case []:
            return "empty"
        case [only]:
            return f"one item: {only}"
        case [first, *rest]:
            return f"starts with {first}, {len(rest)} more after it"


print(summarize([]))
print(summarize([1]))
print(summarize([1, 2, 3, 4]))

### Mapping patterns — matching the shape of a dictionary

A mapping pattern checks that specific keys exist (and, optionally, binds their values), without requiring the dictionary to contain *only* those keys — a deliberately partial match, useful for exactly the kind of loosely structured data that comes from JSON (Intermediate Chapter 55).

In [ ]:
def handle_event(event):
    match event:
        case {"type": "click", "x": x, "y": y}:
            return f"Click at ({x}, {y})"
        case {"type": "keypress", "key": key}:
            return f"Key pressed: {key}"
        case {"type": event_type}:
            return f"Unhandled event type: {event_type}"
        case _:
            return "Malformed event"


print(handle_event({"type": "click", "x": 10, "y": 20}))
print(handle_event({"type": "keypress", "key": "Enter"}))
print(handle_event({"type": "scroll", "amount": 5}))
print(handle_event({}))

Notice `{"type": "click", "x": 10, "y": 20, "timestamp": 123}` would still match the first case, even though the pattern only mentions `type`, `x`, and `y` — a mapping pattern only requires the keys it names to be present with matching values; it doesn't require an exact match of every key, unlike a sequence pattern, which generally does care about overall length unless a `*rest` is used to absorb the remainder deliberately.

### Class patterns — matching an object's type and its attributes together

A class pattern checks that a value is an instance of a given class, while simultaneously extracting specific attributes — genuinely useful once you're pattern-matching over the kind of class hierarchies Intermediate's Part II built.

In [ ]:
from dataclasses import dataclass


@dataclass
class Circle:
    radius: float


@dataclass
class Rectangle:
    width: float
    height: float


def area(shape):
    match shape:
        case Circle(radius=r):
            return 3.14159 * r ** 2
        case Rectangle(width=w, height=h):
            return w * h
        case _:
            raise TypeError(f"Unknown shape: {shape}")


print(area(Circle(radius=3)))
print(area(Rectangle(width=4, height=5)))

`case Circle(radius=r):` checks two things simultaneously: is `shape` an instance of `Circle`, and, if so, bind its `radius` attribute to the name `r`. This reads almost like a more expressive form of `isinstance`, and it composes naturally with dataclasses specifically, since a dataclass's fields are exactly the attributes a class pattern is built to extract.

### Guards — adding a condition beyond the shape itself

A pattern can be refined with an `if` clause — a **guard** — checked only after the pattern itself has already matched, letting you combine structural matching with an ordinary boolean condition.

In [ ]:
def classify(n):
    match n:
        case int() if n < 0:
            return "negative integer"
        case int() if n == 0:
            return "zero"
        case int():
            return "positive integer"
        case float():
            return "a float"
        case _:
            return "not a number"


print(classify(-5))
print(classify(0))
print(classify(5))
print(classify(3.14))

`case int() if n < 0:` first checks that `n` is an `int` (an empty-parentheses class pattern is a plain type check, with nothing to extract), and only proceeds into that branch if the guard, `n < 0`, is *also* true. The order of `case` clauses matters here exactly as it did for Basics' `if`/`elif` chains — Python checks each `case` top to bottom and commits to the first one whose pattern (and guard, if any) succeeds, so `case int() if n == 0:` would never be reached if it were placed after the unconditional `case int():`.

### When `match` is worth it, and when a plain `if` chain remains clearer

`match`/`case` genuinely earns its place once you're checking a value's *shape* — its length, its keys, its type together with its attributes — not merely its equality to a handful of literals. For that latter case (Basics' `if score >= 90: ... elif ...`, or a simple dispatch on a handful of string constants), a plain `if`/`elif` chain remains just as clear, and there's no strong reason to prefer `match` purely for its own sake. The decision worth making deliberately is whether the problem is naturally about *shape* — which is precisely what structural pattern matching was built to express — or simply about a sequence of independent boolean conditions, which `if`/`elif` already expresses perfectly well.

### One problem

> Write a function `process_response(response)` that uses `match`/`case` with mapping patterns to handle a dictionary shaped like `{"status": "success", "data": [...]}`, `{"status": "error", "code": ..., "message": ...}`, and any other shape as an unrecognized response — returning an appropriate descriptive string for each.

In [ ]:
# TODO: define process_response using match/case with mapping patterns


# Part III — Iteration and Asynchronous Python

## 71. Advanced Iterators and Generator Pipelines

Intermediate Chapters 44 through 46 established the iterator protocol, generators, and generator expressions individually. This chapter's subject is what happens when you chain several of them together — because a generator's laziness, established one function at a time in Intermediate Python, compounds in a genuinely useful way once several are connected in sequence.

### Composing generators

A generator function can take an iterable as input and yield a transformed version of it — which means one generator's output can become another generator's input, directly.

In [ ]:
def read_lines(lines):
    for line in lines:
        yield line.strip()


def non_empty(lines):
    for line in lines:
        if line:
            yield line


def as_upper(lines):
    for line in lines:
        yield line.upper()


raw_lines = ["  hello  ", "", "  world  ", "   ", "python"]

pipeline = as_upper(non_empty(read_lines(raw_lines)))

for line in pipeline:
    print(line)

Each function is small, does exactly one transformation, and knows nothing about the others. `pipeline` is not a list — it's a chain of three nested generator objects, none of which has computed anything yet at the point `pipeline` is created. Only the `for` loop's repeated `next()` calls (Intermediate Chapter 33) actually drive execution, and they drive it through all three stages at once, one item at a time.

### Tracing what actually happens for one item

This is worth tracing explicitly, because "one item at a time, through the whole chain" is a genuinely different execution order than "each stage fully processes the whole input before the next stage starts," and confusing the two leads to wrong intuitions about performance and memory.

```text
for line in pipeline:
        ↓
next(pipeline)  -- pipeline is as_upper(...)
        ↓
as_upper asks non_empty for its next item
        ↓
non_empty asks read_lines for its next item
        ↓
read_lines asks raw_lines (the original list) for its next item -- gets "  hello  "
        ↓
read_lines strips it -- yields "hello"
        ↓
non_empty checks it's non-empty -- yields "hello" onward
        ↓
as_upper uppercases it -- yields "HELLO"
        ↓
the for loop receives "HELLO"
```

Notice `read_lines` had to produce **two** raw items before `non_empty` could hand anything onward the first time it was asked for a second value — `"  hello  "` strips to `"hello"` and passes through fine on the very first pull, but if the input started with an empty line, `non_empty` would have to pull *again* from `read_lines`, which would pull again from the original list, before it had anything to yield at all. Nothing runs "all the way through" one stage before starting the next; every stage runs exactly as far as it needs to, to produce exactly one value, and no further, exactly matching the pause-and-resume model Intermediate Chapter 45 established for a single generator, now happening simultaneously across three nested ones.

### Why this matters for large or unbounded input

Because no stage ever materializes an intermediate list, this pipeline uses a small, constant amount of memory regardless of how large `raw_lines` is — the exact payoff Intermediate Chapter 45 demonstrated for a single generator, now compounding across a whole chain of transformations. This is what makes the pattern genuinely valuable for **streaming data**: reading a multi-gigabyte log file line by line, filtering and transforming as you go, without ever holding the whole file in memory at once.

In [ ]:
def take(iterable, n):
    for i, item in enumerate(iterable):
        if i >= n:
            return
        yield item


def integers_from(start):
    n = start
    while True:
        yield n
        n += 1


def multiples_of(n, numbers):
    for number in numbers:
        if number % n == 0:
            yield number


first_five_multiples_of_seven = take(multiples_of(7, integers_from(1)), 5)
print(list(first_five_multiples_of_seven))

`integers_from(1)` is an infinite generator — it never raises `StopIteration` on its own. This would be an infinite loop if you ever tried to `list()` it directly. But piped through `multiples_of` and then `take`, only exactly the values actually needed are ever computed: `take` stops pulling after five items, which means `multiples_of` never gets asked for a sixth, which means `integers_from` never has to produce past `35`. The infinite generator is entirely safe here specifically because nothing downstream ever asks it for more than it can supply lazily, on demand.

### `itertools` — a standard-library toolbox built on exactly this idea

The pattern above — `take`, chaining, filtering — is common enough that the standard library provides tested, general versions of the most frequent cases, in the `itertools` module.

In [ ]:
import itertools

first_five = itertools.islice(itertools.count(1), 5)
print(list(first_five))

evens_only = itertools.filterfalse(lambda n: n % 2, range(20))
print(list(evens_only))

chained = itertools.chain([1, 2], [3, 4], [5, 6])
print(list(chained))

`itertools.count(1)` is exactly `integers_from(1)` from above, already written and tested; `itertools.islice` is exactly `take`. Writing your own versions first, as this chapter did, is worth doing once to genuinely understand the mechanism — but in real code, reaching for `itertools` rather than reimplementing these specific patterns is almost always the better choice, precisely because they're already correct, already fast, and immediately recognizable to any other Python developer reading the code.

### One problem

> Using a generator pipeline (your own functions, or `itertools`, or a mix), process an infinite sequence of natural numbers to lazily produce the first ten Fibonacci numbers that are also even, without ever computing the full Fibonacci sequence or the full set of even numbers up front.

In [ ]:
import itertools

# TODO: build a lazy pipeline producing the first ten even Fibonacci numbers


## 72. Async Programming — `async` and `await`

Before any `async` syntax at all, it's worth being precise about the actual problem this part of the book exists to solve, because "async" is frequently introduced through its syntax before the problem it addresses is ever stated clearly.

### Blocking, and what it costs

Consider a function that fetches data from three different network sources, one after another: 

In [ ]:
import time


def fetch_data(source, delay):
    print(f"Starting fetch from {source}")
    time.sleep(delay)   # standing in for a real network request
    print(f"Finished fetch from {source}")
    return f"data from {source}"


start = time.perf_counter()

fetch_data("server A", 1)
fetch_data("server B", 1)
fetch_data("server C", 1)

print(f"Total time: {time.perf_counter() - start:.2f} seconds")

Roughly three seconds, because each `time.sleep` genuinely **blocks** — it stops the entire program from doing anything at all, including starting the next fetch, until that specific wait is over. This is worth naming precisely: a **blocking operation** is one that occupies the thread of execution for its whole duration, even though, for a real network request, the CPU isn't actually *doing* anything during most of that time — it's simply waiting for a response to arrive.

That's the actual inefficiency async programming targets: while `fetch_data("server A", 1)` is waiting for its network response, nothing stops the program from *also* starting the request to server B during that same waiting period — the two waits could genuinely overlap, since neither one needs the CPU while it waits. A program written with ordinary, blocking calls has no way to express "start this, and while it's waiting, go start that other thing too." Async syntax exists specifically to express exactly that.

### Concurrency versus parallelism — a distinction worth making precisely before any code

These two words are often used loosely as synonyms, and for the material ahead, conflating them will actively mislead you.

**Concurrency** means structuring a program so that multiple tasks can be *in progress* at overlapping times — not necessarily running at the literal same instant, but interleaved, with the program able to switch between them while one is waiting. **Parallelism** means multiple tasks are *actually executing at the same physical instant*, which requires genuinely separate hardware execution units — multiple CPU cores, specifically.

```text
concurrency:  task A waits  →  task B runs  →  task A resumes  →  task B waits  →  ...
              (one core, interleaved -- never two things truly "at once")

parallelism:  task A runs  ─┐
              task B runs  ─┴─  genuinely simultaneous, on separate cores
```

Async programming, as covered in this book, is a tool for **concurrency**, not parallelism — it lets a single thread interleave many waiting tasks efficiently, but it does not, by itself, use more than one CPU core. That claim will be made precise and defended in Chapter 78, once the GIL has been properly explained; for now, hold it as a statement to return to, not yet a fully justified one.

### Coroutines — functions that can pause

`async def` defines a **coroutine function** — calling it does not run its body immediately, in a close parallel to how calling a generator function (Intermediate Chapter 45) doesn't run its body immediately either.

In [ ]:
async def greet(name):
    print(f"Hello, {name}")
    return f"Greeted {name}"


result = greet("Ali")
print(result)
print(type(result))

Calling `greet("Ali")` produced neither the print statement nor the return value — it produced a **coroutine object**, a paused, not-yet-started unit of work, exactly parallel to how calling a generator function produces a paused generator object rather than running anything. To actually run it, it needs to be **awaited**.

In [ ]:
result = await greet("Ali")
print(result)

`await` is what actually drives a coroutine's execution — the direct counterpart to `next()` driving a generator, though the underlying machinery (covered fully in the next chapter) is somewhat more elaborate, because a coroutine can pause specifically to let *other* coroutines run during its wait, which a generator's `next()` was never designed to coordinate.

### A note on running this in a notebook versus a script

This cell used `await greet("Ali")` directly, without wrapping it in `asyncio.run(...)`. That's a genuine convenience specific to notebook environments: Jupyter already runs its own event loop internally, and allows `await` at the top level of a cell as a result. In an ordinary `.py` script, top-level `await` is not legal syntax at all — a script needs an explicit entry point, using `asyncio.run(...)`, which Chapter 74 covers properly. Both express the identical idea — "actually drive this coroutine to completion" — the difference is purely about what kind of program is doing the driving.

### The actual payoff — overlapping the waits

Here's the version of the original three-fetch example that actually achieves what blocking code structurally cannot: 

In [ ]:
import asyncio


async def fetch_data(source, delay):
    print(f"Starting fetch from {source}")
    await asyncio.sleep(delay)   # a non-blocking wait -- this is the key difference
    print(f"Finished fetch from {source}")
    return f"data from {source}"


async def main():
    start = time.perf_counter()

    results = await asyncio.gather(
        fetch_data("server A", 1),
        fetch_data("server B", 1),
        fetch_data("server C", 1),
    )

    print(f"Total time: {time.perf_counter() - start:.2f} seconds")
    return results


results = await main()
print(results)

Roughly one second total, not three. `asyncio.sleep`, unlike `time.sleep`, doesn't block the whole program — it tells the underlying scheduler (Chapter 73's subject) "this coroutine has nothing to do for one second; feel free to run something else in the meantime." `asyncio.gather` starts all three fetches essentially at once and lets their waits overlap, which is precisely the capability ordinary blocking calls never offered. The three `"Starting fetch"` messages all print immediately, back to back, before any `"Finished fetch"` message appears — visible, direct evidence that all three waits are genuinely happening concurrently, on a single thread, rather than one after another.

### One problem

> Write three coroutine functions simulating downloading three different files, each taking a different, explicit delay via `asyncio.sleep`. Run them sequentially with individual `await` calls and measure the time; then run them concurrently with `asyncio.gather` and measure again, printing both durations for comparison.

In [ ]:
import asyncio
import time

# TODO: define the three coroutines, then compare sequential vs. concurrent timing


## 73. Coroutines and the Event Loop

The last chapter showed *that* `asyncio.gather` overlaps waits. This chapter explains *how* — the specific mechanism that lets one single thread interleave several coroutines without ever running more than one line of Python at any given instant.

### The event loop, conceptually

An **event loop** is a running piece of machinery whose entire job is: keep a collection of paused coroutines, and repeatedly pick one that's ready to make progress, run it until it pauses again (or finishes), and move on to the next ready one. It's a scheduler, in the same general sense an operating system's process scheduler is — deciding what runs next — except it operates entirely within a single Python thread, and it never interrupts a coroutine that hasn't explicitly agreed to be interrupted.

### Suspension points — where control can actually change hands

This last detail is the single most important thing to understand about `asyncio`, because it directly contradicts what people sometimes assume when they first meet the word "concurrency." A coroutine's code runs completely normally, one line after another, exactly like ordinary synchronous code — **right up until it hits an `await`**. Only at an `await` does the coroutine voluntarily hand control back to the event loop, which can then choose to run a different coroutine for a while.

```text
coroutine A                event loop                coroutine B
    │
    │  runs normally
    │
    ├─── await ──────────►  picks another ready
    │    (A suspends)       coroutine to run
    │                            │
    │                            ├─── runs B normally
    │                            │
    │                            ├─── await ──────► (B suspends)
    │                            │
    │  ◄──── resumes A ──────────┘
    │  (once A's wait is done and the loop gets back to it)
    │
    │  continues running
```

This is **cooperative multitasking**: coroutines aren't forcibly interrupted by the system at arbitrary points — they cooperate by explicitly yielding control at `await` boundaries, and nowhere else. This has a direct, practical consequence worth stating precisely: a long stretch of ordinary, CPU-bound Python code inside a coroutine, with no `await` anywhere in it, will run to completion without ever letting any other coroutine make progress, no matter how many are waiting — cooperative multitasking only helps at the points where a coroutine chooses to pause.

### Tracing a small program step by step

This is worth watching directly rather than only reading about, using explicit print statements as markers of exactly when each piece of code actually runs.

In [ ]:
import asyncio


async def task(name, delay):
    print(f"{name}: starting")
    await asyncio.sleep(delay)
    print(f"{name}: resumed after {delay}s")


async def main():
    print("main: about to start both tasks")
    await asyncio.gather(
        task("A", 2),
        task("B", 1),
    )
    print("main: both tasks finished")


await main()

Read the actual order of output, not the order the code is written in: `"A: starting"` and `"B: starting"` print back to back, immediately — both coroutines run up to their first `await` before either one actually pauses. Then, roughly one second later, `"B: resumed after 1s"` prints — B's shorter wait finishes first, so the event loop resumes B specifically, even though A was started first. Only after another second does `"A: resumed after 2s"` print. The event loop is not running these in the textual order they were written or started; it's running whichever coroutine is *ready* at any given moment, and a coroutine only becomes ready again once whatever it was awaiting (here, a timer) has actually completed.

### An explicit, honest statement about cores

Everything shown in this chapter has happened on a single thread of a single process, the entire time. `asyncio.gather` did not distribute `task("A", 2)` and `task("B", 1)` onto separate CPU cores — there is exactly one Python call stack actively executing at any instant, switching between the two coroutines only at their `await` points. This is worth restating precisely because the visible speed-up (the fetches from Chapter 72 finishing in roughly one second rather than three) can look, superficially, like parallel execution — it is not. It's concurrency: overlapping *waiting*, not overlapping *computing*. If both tasks had been genuinely CPU-intensive rather than waiting on a timer or a network response, `asyncio` alone would provide no benefit at all — a subject Chapter 78 will make fully precise once the GIL itself has been explained.

### One problem

> Write three coroutines, each printing a start message, awaiting `asyncio.sleep` with a different delay, then printing a finish message with the elapsed time since the program started. Run them with `asyncio.gather` and, based purely on reading the printed output (not the source order), explain in a comment which one the event loop resumed first, second, and third, and why.

In [ ]:
import asyncio
import time

# TODO: define the three coroutines and observe the actual resume order


## 74. `asyncio` Tasks, Futures, and Scheduling

Chapters 72 and 73 used `asyncio.gather` to run coroutines concurrently without examining exactly what `gather` does underneath. This chapter opens that up, and covers the pieces needed to actually manage concurrent work deliberately — starting it, cancelling it, timing it out, and handling its failures.

### Running a program's entry point — `asyncio.run`

Every example so far in this notebook has used top-level `await`, available specifically because Jupyter already runs an event loop. A real Python script has no such loop running automatically — it needs to create and manage one explicitly, and `asyncio.run(coroutine)` is the standard way to do that: it creates a fresh event loop, runs the given coroutine (and anything it awaits) to completion, and closes the loop afterward. It cannot be called from inside code that's already running inside an event loop — calling it from within this notebook directly would raise a `RuntimeError`, which is precisely why the notebook's own examples use plain `await` instead. To show `asyncio.run` behaving exactly as it would in a genuine script, this chapter writes one to disk and runs it as a separate process.

In [ ]:
script = '''
import asyncio


async def main():
    print("running inside asyncio.run")
    await asyncio.sleep(0.2)
    print("done")


if __name__ == "__main__":
    asyncio.run(main())
'''

with open("async_script.py", "w") as f:
    f.write(script)

In [ ]:
import subprocess

result = subprocess.run(["python", "async_script.py"], capture_output=True, text=True)
print(result.stdout)

That's the shape every real, standalone async Python program starts with: one `async def main():` and exactly one `asyncio.run(main())` at the bottom, guarded by `if __name__ == "__main__":` — the same guard from Intermediate Chapter 63's discussion of avoiding problems when a module is imported rather than run directly.

### Tasks — scheduling a coroutine to run in the background

`await coroutine` runs a coroutine and waits for it, in order, right there. `asyncio.create_task(coroutine)` does something meaningfully different: it schedules the coroutine to start running *now*, in the background, and immediately gives you back a `Task` object you can continue on without waiting for — the actual waiting, if you want it, happens later, separately.

In [ ]:
import asyncio


async def worker(name, delay):
    await asyncio.sleep(delay)
    return f"{name} done"


async def main():
    task_a = asyncio.create_task(worker("A", 2))
    task_b = asyncio.create_task(worker("B", 1))

    print("both tasks scheduled, main continues immediately")

    result_a = await task_a
    result_b = await task_b

    print(result_a, "/", result_b)


await main()

`"both tasks scheduled, main continues immediately"` prints right away, before either worker has finished — `create_task` doesn't wait; it hands control back to `main` immediately, while the event loop has already begun running both workers concurrently in the background. The later `await task_a` and `await task_b` calls don't *start* anything (both are already running) — they simply pause `main` until each one's result is actually available. This is, in fact, exactly what `asyncio.gather` does for you as a convenience: it creates tasks for each coroutine given to it and awaits all of them together.

### Futures — the general concept a `Task` is built on

A **future** is a lower-level object representing a value that isn't ready yet but will be at some point — a placeholder for a result. A `Task` is, specifically, a future that wraps a coroutine and drives it to completion via the event loop. You'll rarely construct a raw future directly in ordinary application code, but the concept is worth naming, because `Task`, `asyncio.gather`'s return values, and several other pieces of `asyncio` are all, underneath, variations on this same "not-yet-available result" idea.

### Cancellation

A task that's no longer needed can be cancelled explicitly, and the coroutine it wraps receives a specific exception, `asyncio.CancelledError`, at whatever `await` point it happens to be suspended on — a coroutine cannot be cancelled at an arbitrary point mid-line, only at a suspension point, exactly consistent with Chapter 73's description of cooperative multitasking.

In [ ]:
import asyncio


async def long_task():
    try:
        print("starting long task")
        await asyncio.sleep(10)
        print("this line never runs")
    except asyncio.CancelledError:
        print("long task was cancelled")
        raise


async def main():
    task = asyncio.create_task(long_task())
    await asyncio.sleep(0.5)
    task.cancel()
    try:
        await task
    except asyncio.CancelledError:
        print("main observed the cancellation too")


await main()

`task.cancel()` doesn't stop `long_task` instantly and silently — it raises `CancelledError` inside `long_task`, right at its current suspension point (inside `asyncio.sleep(10)`, here), giving the coroutine a chance to clean up (the `except` block, printing its own message) before the exception propagates further. The `raise` inside that `except` block matters: swallowing a `CancelledError` without re-raising it is generally considered incorrect — the caller genuinely asked for cancellation, and hiding that from the rest of the program can leave a task looking like it finished normally when it was actually cut short.

### Timeouts

Waiting indefinitely for something that might never complete is rarely the right default. `asyncio.timeout` (Python 3.11+) provides a clean way to bound how long a piece of code is allowed to wait.

In [ ]:
import asyncio


async def slow_operation():
    await asyncio.sleep(5)
    return "finally done"


async def main():
    try:
        async with asyncio.timeout(1):
            result = await slow_operation()
            print(result)
    except TimeoutError:
        print("gave up waiting after 1 second")


await main()

`async with asyncio.timeout(1):` (an **async context manager**, the next chapter's proper subject) cancels whatever's running inside the block if it hasn't completed within the given number of seconds, raising a plain `TimeoutError` for the surrounding code to handle — a considerably cleaner pattern than manually tracking elapsed time and cancelling a task by hand.

### Handling exceptions from concurrent tasks

If one of several tasks passed to `asyncio.gather` raises an exception, by default `gather` immediately re-raises it to the caller — but the other tasks keep running in the background regardless, which is worth being aware of, since it means an unhandled exception from one task doesn't automatically stop the others: 

In [ ]:
import asyncio


async def worker(name, delay, fail=False):
    await asyncio.sleep(delay)
    if fail:
        raise ValueError(f"{name} failed")
    return f"{name} succeeded"


async def main():
    try:
        results = await asyncio.gather(
            worker("A", 1),
            worker("B", 0.5, fail=True),
            worker("C", 1.5),
        )
        print(results)
    except ValueError as e:
        print(f"gather raised: {e}")


await main()

`asyncio.gather` also accepts `return_exceptions=True`, which changes this behavior — instead of raising immediately, it collects each task's outcome (result or exception) into the returned list, letting you inspect every task's result individually rather than having the first failure short-circuit the rest.

### One problem

> Write a coroutine `fetch(url, delay, fail_for)` that simulates a network request, raising a `ValueError` if `url == fail_for`, otherwise succeeding after `delay` seconds. Use `asyncio.gather` with `return_exceptions=True` across four different simulated URLs (one of which should fail), and print which ones succeeded and which failed, along with their results or error messages.

In [ ]:
import asyncio

# TODO: define fetch(url, delay, fail_for) and gather results with return_exceptions=True


## 75. Async Context Managers and Iterators

Intermediate Chapter 50 established `__enter__`/`__exit__` and Intermediate Chapter 33 and 44 established the iterator protocol, both entirely synchronous. Once resources involve waiting — an asynchronous database connection that needs to await its own setup, a network stream that yields chunks as they arrive — both protocols need an asynchronous counterpart, and Python provides one for each, deliberately mirroring the synchronous versions as closely as possible.

### `async with` — a context manager whose setup and teardown can await

Chapter 74 already used `async with asyncio.timeout(1):` without pausing on the syntax itself. Here's what makes a context manager usable with `async with`, built directly, mirroring Intermediate Chapter 50's `Announce` class exactly, one layer further.

In [ ]:
import asyncio


class AsyncConnection:
    def __init__(self, name):
        self.name = name

    async def __aenter__(self):
        print(f"opening connection to {self.name}")
        await asyncio.sleep(0.3)   # standing in for a real async handshake
        print(f"connected to {self.name}")
        return self

    async def __aexit__(self, exc_type, exc_value, traceback):
        print(f"closing connection to {self.name}")
        await asyncio.sleep(0.1)
        print(f"connection to {self.name} closed")


async def main():
    async with AsyncConnection("database") as conn:
        print(f"using {conn.name}")


await main()

`__aenter__` and `__aexit__` are exactly `__enter__` and `__exit__`, except declared with `async def`, meaning each one returns a coroutine rather than a plain value, and `async with` automatically `await`s each one for you. The entire reason this exists, rather than reusing plain `with`, is that opening a real network connection or database handle often genuinely needs to *wait* — a synchronous `__enter__` has no way to pause and let other coroutines run during that wait, exactly the limitation Chapter 72 opened with; `__aenter__` does.

The guarantee is identical to the synchronous version too: `__aexit__` runs regardless of whether the block completed normally or raised — Intermediate Chapter 50's cleanup guarantee, carried over unchanged, just with the ability for that cleanup itself to `await` something.

### `async for` — iterating over values that arrive over time

The synchronous iterator protocol (Intermediate Chapter 44) requires `__iter__` and `__next__`. Its asynchronous counterpart requires `__aiter__` and `__anext__`, and `async for` is the syntax that drives it, calling and awaiting each in turn.

In [ ]:
import asyncio


class Countdown:
    def __init__(self, start):
        self.current = start

    def __aiter__(self):
        return self

    async def __anext__(self):
        if self.current <= 0:
            raise StopAsyncIteration
        await asyncio.sleep(0.2)
        value = self.current
        self.current -= 1
        return value


async def main():
    async for n in Countdown(3):
        print(n)


await main()

This mirrors Intermediate Chapter 44's hand-built `CountUpTo` class almost exactly — `__aiter__` plays the role `__iter__` did, `__anext__` plays the role `__next__` did, and `StopAsyncIteration` plays the role `StopIteration` did. The genuine difference is that `__anext__` is a coroutine, so producing "the next value" can itself involve waiting — for the next row of a database query result, the next chunk of a streamed HTTP response — without blocking anything else that might be running concurrently while it waits.

### Async generators — the lazy, concise version of the same idea

Exactly as an ordinary generator function (`yield` inside `def`) is usually far more convenient than hand-writing a full iterator class, an **async generator** (`yield` inside `async def`) is the equivalent shortcut for the asynchronous iterator protocol.

In [ ]:
async def countdown(start):
    current = start
    while current > 0:
        await asyncio.sleep(0.2)
        yield current
        current -= 1


async def main():
    async for n in countdown(3):
        print(n)


await main()

This behaves identically to the hand-written `Countdown` class above, for the same reason Intermediate Chapter 45's `yield`-based generators replaced hand-written iterator classes: the language handles constructing `__aiter__` and `__anext__` (and raising `StopAsyncIteration` automatically once the function body finishes) for you, leaving just the actual logic — a `while` loop with a `yield` in it — visible.

### Connecting this back to where the chapter started

Nothing here introduced a new *idea* — every piece of asynchronous machinery in this chapter is a direct, deliberate mirror of a synchronous concept you already had from Intermediate Python, with `async`/`await` inserted exactly at the points where real waiting needs to happen without blocking anything else. That mirroring is not a coincidence of naming; it's the actual design of the feature, meant to let existing intuitions about context managers and iteration transfer directly into asynchronous code, rather than requiring an entirely separate mental model.

### One problem

> Write an async generator `paginated_results(pages, delay)` that simulates fetching pages of data one at a time (each page a list of a few strings), waiting `delay` seconds before yielding each page. Use `async for` to consume it and print each page as it arrives, along with a running total of items seen so far.

In [ ]:
import asyncio

# TODO: define paginated_results as an async generator and consume it with async for


# Part IV — Concurrency and Parallelism

## 76. Threads and Thread-Based Concurrency

Part III's concurrency was cooperative — a single thread, switching between coroutines only at explicit `await` points, entirely under the program's own control. This chapter introduces a different mechanism for concurrency, one where the switching is no longer something the program deliberately negotiates.

### What a thread actually is

A **thread** is an independent sequence of execution within the same process, sharing that process's memory with every other thread in it. This sharing is the single most important fact about threads, and it cuts both ways: it makes passing data between threads essentially free (no copying, no serialization — they're just looking at the same objects), and it's also the exact source of every bug this chapter and Chapter 79 are about.

```text
process
  ├── thread 1  ─┐
  ├── thread 2  ─┼──►  all sharing the same memory
  └── thread 3  ─┘
```

Unlike the coroutines from Part III, the operating system — not your program — decides when to pause one thread and let another run, and it can do this at essentially any point, not just at points your code explicitly marks as safe. This is worth holding in contrast to Chapter 73's cooperative multitasking: threads are **preemptive** — a thread can be interrupted between almost any two instructions, without its consent or awareness.

### Starting and joining threads

```python
threading.Thread(target=function, args=(...))
```

creates a thread that will run `function(...)` once started.

In [ ]:
import threading
import time


def worker(name, delay):
    print(f"{name}: starting")
    time.sleep(delay)
    print(f"{name}: done")


start = time.perf_counter()

t1 = threading.Thread(target=worker, args=("Thread-A", 1))
t2 = threading.Thread(target=worker, args=("Thread-B", 1))

t1.start()
t2.start()

t1.join()
t2.join()

print(f"Total time: {time.perf_counter() - start:.2f} seconds")

Roughly one second, not two — both threads' `time.sleep` calls overlap, because each thread waits independently, and waiting doesn't require the CPU. `.start()` begins a thread running in the background; `.join()` blocks the calling thread until the target thread has finished. Without both `.join()` calls, `main` (the original thread) could reach the final `print` before either worker thread had actually finished — `.join()` is what enforces "wait for this thread before continuing," analogous to `await task` for an `asyncio` task in the previous part.

### Where threads genuinely help: I/O-bound work

The overlap above worked for exactly the same underlying reason `asyncio.gather` overlapped waits in Chapter 72: `time.sleep`, like a real network request or disk read, doesn't need the CPU while it's waiting — it's **I/O-bound** work, and while one thread waits, the operating system is free to run another thread that has actual work to do (or that's also just waiting, in which case both simply wait concurrently, at no real cost). This is the situation where threading is a legitimate, effective concurrency tool.

### Where threads do not help — and the direct warning this chapter promised

Try the equivalent with work that genuinely occupies the CPU rather than waiting for anything: 

In [ ]:
def cpu_heavy_work(n):
    total = 0
    for i in range(n):
        total += i * i
    return total


start = time.perf_counter()
cpu_heavy_work(20_000_000)
cpu_heavy_work(20_000_000)
print(f"Sequential: {time.perf_counter() - start:.2f} seconds")

start = time.perf_counter()
t1 = threading.Thread(target=cpu_heavy_work, args=(20_000_000,))
t2 = threading.Thread(target=cpu_heavy_work, args=(20_000_000,))
t1.start()
t2.start()
t1.join()
t2.join()
print(f"Threaded: {time.perf_counter() - start:.2f} seconds")

Run this and compare the two times honestly: the threaded version is not meaningfully faster than the sequential one, and depending on the machine, it can even be slightly slower once thread-management overhead is included. This is the direct, empirical demonstration of a claim this chapter is making deliberately, ahead of Chapter 78's full explanation: in CPython specifically, threads do not speed up CPU-bound work, because of a mechanism — the GIL — that ensures only one thread executes Python bytecode at any given instant, no matter how many threads exist. Threading genuinely helps when threads spend their time *waiting*; it does essentially nothing for threads that spend their time *computing*. Holding onto this distinction now will make Chapter 78's explanation land as confirmation of something you've already observed, rather than an abstract rule to memorize.

### Race conditions — the cost of shared memory

Because threads share memory, two threads modifying the same value at overlapping times can interfere with each other in a way that produces a wrong result, silently, without either thread's code containing any visible mistake.

In [ ]:
counter = 0


def increment(times):
    global counter
    for _ in range(times):
        counter += 1


threads = [threading.Thread(target=increment, args=(100_000,)) for _ in range(4)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print(counter)
print("Expected:", 4 * 100_000)

Run this more than once, and the printed value likely won't match the expected total consistently — and it may not even be the *same* wrong number each run. `counter += 1` looks like one atomic step but is actually at least three: read `counter`'s current value, add one, write the result back. If a thread gets preempted between the read and the write, another thread can read the same "old" value, increment it, write it back — and then the first thread resumes and overwrites that update with its own, now-stale calculation, silently discarding one of the increments. This is a **race condition**: correctness depends on the precise, unpredictable timing of when threads happen to be interrupted, which means the same code can behave correctly nearly every time and then fail unpredictably, which is exactly what makes race conditions among the hardest class of bugs to reliably reproduce and debug. Chapter 79 introduces the tool — a lock — built specifically to prevent this.

### One problem

> Write a function that simulates downloading five files with I/O-bound delays (using `time.sleep`), and compare the total time using threads versus running them sequentially, printing both durations. Then write a second comparison using a CPU-bound function (like the sum-of-squares example above) and show that threading provides little or no benefit there, printing both durations for that case too.

In [ ]:
import threading
import time

# TODO: compare threaded vs sequential timing for both an I/O-bound and a CPU-bound task


## 77. Processes and Multiprocessing

The last chapter's CPU-bound example showed threads providing no real speedup. This chapter introduces the tool that actually does provide one for that kind of work, at a genuinely different cost.

### A process, versus a thread

A **process** is a separate, independent instance of a running program, with its **own** memory space — nothing shared automatically, unlike threads within the same process.

```text
process 1                    process 2
  memory A        ✗ no automatic sharing ✗        memory B
```

This is the fundamental trade against threads: processes cannot interfere with each other's data by accident, because there's no shared data to interfere with — but by the same token, sharing data between processes deliberately requires explicit communication (typically involving serialization, echoing Intermediate Chapter 55's discussion of converting data to a transferable form), which threads never needed at all.

### Why processes bypass the limitation threads hit

Each process gets its own independent Python interpreter, and — this is the detail that matters for Chapter 78's subject — its own independent GIL. Two processes running CPU-bound Python code genuinely run on separate CPU cores at the same physical instant; nothing inside one process's interpreter restricts what a completely separate process's interpreter is doing.

In [ ]:
def cpu_heavy_work(n):
    total = 0
    for i in range(n):
        total += i * i
    return total

In [ ]:
import multiprocessing as mp
import time

print(f"CPU cores available: {mp.cpu_count()}")

start = time.perf_counter()
cpu_heavy_work(20_000_000)
cpu_heavy_work(20_000_000)
print(f"Sequential: {time.perf_counter() - start:.2f} seconds")

start = time.perf_counter()
with mp.Pool(2) as pool:
    pool.map(cpu_heavy_work, [20_000_000, 20_000_000])
print(f"Multiprocessing: {time.perf_counter() - start:.2f} seconds")

On a machine with at least two CPU cores actually available to it, this runs close to twice as fast as the sequential version — the two calls to `cpu_heavy_work` genuinely execute simultaneously, on separate cores, in separate processes, with no GIL shared between them to serialize their execution. The `mp.cpu_count()` line matters more than it looks: if it reports `1` — a single-core machine, or a container deliberately restricted to one core, as some cloud and sandboxed environments are — a `Pool(2)` still creates two worker processes, but the operating system has only one core to actually run them on, so they take turns exactly as threads would, and the speedup disappears entirely, or even turns into a small slowdown once process-creation overhead is included. This is worth checking directly rather than assuming: multiprocessing's benefit is bounded by genuinely available cores, not by how many worker processes you ask for.

### `Pool` — managing a group of worker processes

`multiprocessing.Pool(n)` creates `n` worker processes upfront and reuses them across multiple tasks, rather than paying the (nontrivial) cost of starting a brand-new process for every single unit of work. `.map(function, iterable)` distributes each item of `iterable` to an available worker, exactly mirroring the built-in `map()` from Intermediate Chapter 56, but executing genuinely in parallel rather than sequentially and lazily on one thread.

### The real cost: serialization

Every argument sent to a worker process, and every result sent back, has to cross the boundary between separate memory spaces — which means it has to be **serialized** (Intermediate Chapter 55's `pickle`, specifically, is what `multiprocessing` uses by default) on one side and deserialized on the other.

In [ ]:
def process_large_data(data):
    return sum(x * x for x in data)


large_lists = [list(range(1_000_000)) for _ in range(4)]

start = time.perf_counter()
results_sequential = [process_large_data(d) for d in large_lists]
print(f"Sequential: {time.perf_counter() - start:.2f} seconds")

start = time.perf_counter()
with mp.Pool(4) as pool:
    results_parallel = pool.map(process_large_data, large_lists)
print(f"Multiprocessing: {time.perf_counter() - start:.2f} seconds")

print(results_sequential == results_parallel)

Depending on the size of the data being passed and the amount of actual computation each worker does with it, the overhead of pickling large arguments and results across process boundaries can meaningfully eat into, or even exceed, the benefit of running in parallel at all. This is worth testing rather than assuming: multiprocessing pays off clearly when each unit of work involves substantial computation relative to the size of data being passed in and out, and pays off far less — sometimes negatively — when the data being shuffled between processes is large relative to how much work is actually done with it.

### Not every workload benefits — and processes are not a universal upgrade over threads

Reapply the last chapter's I/O-bound example (downloading files, waiting on `time.sleep`) using processes instead of threads, and you'll find no meaningful advantage over the threaded version — and a real cost in the overhead of starting separate processes, each with its own full Python interpreter, for work that was never actually limited by the GIL in the first place. Processes solve the CPU-bound problem threads cannot; they do not make I/O-bound work any faster than threads already made it, and they're considerably heavier to create and communicate with. Chapter 80 turns this into an explicit decision framework; for now, the important habit is asking, honestly, whether a workload is actually CPU-bound before reaching for `multiprocessing`.

### One problem

> Write a CPU-bound function that checks whether each number in a list is prime (by trial division), and compare the total time to check primality for a list of 8 large numbers sequentially versus using a `multiprocessing.Pool` with 4 workers, printing both durations.

In [ ]:
import multiprocessing as mp
import time

# TODO: define is_prime, then compare sequential vs multiprocessing timing


## 78. The GIL and What It Actually Means

Chapters 76 and 77 built toward this chapter empirically: threads didn't speed up CPU-bound work; processes did. This chapter names, precisely, the mechanism responsible for that difference, and is deliberately careful about which claims belong to Python the language versus CPython the specific interpreter almost everyone actually runs.

### What the GIL is

The **Global Interpreter Lock** is a single lock, held by the CPython interpreter, that only one thread may hold at a time — and a thread must hold it to execute Python bytecode at all. This is a **CPython implementation detail**, not a requirement of the Python language itself; other implementations of Python (Jython, IronPython, and — as an active, ongoing effort at the time of this writing — a "free-threaded" build of CPython itself, without a GIL) make different choices. Everything concrete in this chapter describes CPython's standard behavior, which is what you are almost certainly running.

### What it actually protects

CPython's internal memory management — reference counting, covered properly in Chapter 81 — is not, on its own, safe if two threads could modify the same object's internal bookkeeping at the exact same instant. The GIL exists specifically to prevent that: by ensuring only one thread runs Python bytecode at a time, CPython avoids needing far more fine-grained (and far more complex and slower for the single-threaded case) locking around every single object's internals.

### What it prevents

Two Python threads within the same process can never execute Python bytecode at the literal same instant, no matter how many CPU cores the machine has. This is the exact, precise fact behind Chapter 76's empirical result: `t1` and `t2`, both running `cpu_heavy_work`, were never actually computing simultaneously — the GIL was handing control back and forth between them, but always to only one at a time, which is why running them "concurrently" via threads took no less time than running them one after another.

### What it does not prevent

This is the part most often stated wrong, and worth being exact about, because the correct version directly explains Chapter 76's *other* result. The GIL is released — deliberately, by CPython itself — whenever a thread is waiting on something outside the interpreter: a network call, a file read, `time.sleep`, and similar I/O operations. During that release, a different thread is completely free to acquire the GIL and run actual Python bytecode. This is precisely why threading helped the I/O-bound example: each thread's `time.sleep` released the GIL for its entire duration, leaving it free for the other thread to run during that exact window.

```text
CPU-bound threads:  thread A runs ── [GIL held the whole time] ── thread A yields briefly ── thread B runs ── ...
                    (bytecode execution is always serialized -- no real overlap in actual computation)

I/O-bound threads:  thread A starts a wait ── [GIL released] ── thread B runs freely during A's wait ── ...
                    (genuine overlap, because A isn't using the interpreter at all during its wait)
```

### Processes, restated precisely in light of this

Each process launched via `multiprocessing` (Chapter 77) runs its own separate CPython interpreter, and therefore has its **own, separate GIL** — entirely independent of any other process's. That's the exact, complete reason two processes can run CPU-bound Python code truly simultaneously on separate cores: there is no single, shared lock between them to serialize their execution the way there is between threads in the same process.

### Why "Python has no concurrency" is a wrong way to describe this

The GIL is a real, meaningful constraint on *parallelism* for CPU-bound work using threads specifically, within a single process. It says nothing at all about concurrency in general — Part III's entire `asyncio` model achieves genuine, useful concurrency for I/O-bound work with no threads and no GIL contention involved at all (a single thread, cooperatively switching between coroutines), and threading itself remains a completely legitimate concurrency tool for I/O-bound work, exactly as Chapter 76 demonstrated. "Python has no concurrency" conflates concurrency in general with one specific mechanism (multi-core parallelism via threads) that the GIL specifically constrains — three chapters of this book have now demonstrated, empirically, that Python offers several distinct, genuinely useful concurrency tools, each suited to a different kind of workload, which is precisely Chapter 80's subject.

### One problem

> Using the `threading` module, start two threads: one running a tight CPU-bound loop, and one printing the current time once per second for five seconds. Observe and explain, in a comment, whether the timer thread's printouts stay on schedule or are noticeably delayed by the CPU-bound thread — and connect that observation explicitly to what this chapter said about the GIL only being released at specific points.

In [ ]:
import threading
import time

# TODO: run a CPU-bound thread and a once-per-second timer thread together, then observe


## 79. Queues, Locks, Events, and Synchronization

Chapter 76 ended with a race condition — `counter += 1`, executed by four threads, silently losing updates because the operation isn't atomic. This chapter covers the standard tools for making shared, concurrent access to data actually safe.

### `Lock` — enforcing mutual exclusion

A **lock** (sometimes called a mutex, short for "mutual exclusion") is an object that at most one thread can hold at a time; any other thread attempting to acquire it simply waits until the current holder releases it.

In [ ]:
import threading

counter = 0
lock = threading.Lock()


def increment(times):
    global counter
    for _ in range(times):
        with lock:
            counter += 1


threads = [threading.Thread(target=increment, args=(100_000,)) for _ in range(4)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print(counter)
print("Expected:", 4 * 100_000)

Now the result is reliably correct, every run. `with lock:` (a context manager, exactly the protocol from Intermediate Chapter 50) ensures that the read-increment-write sequence inside it happens as an uninterruptible unit from every other lock-respecting thread's point of view — if thread A is inside the `with lock:` block, thread B's attempt to enter its own `with lock:` block simply blocks until A releases it. This is the direct fix for the race condition Chapter 76 demonstrated: the *operation*, not just the individual line, is now protected as a whole.

### Deadlocks — the cost locks can introduce

A lock solves the race-condition problem, but it introduces its own class of bug. A **deadlock** occurs when two or more threads each hold a lock the other one is waiting for, and neither can ever proceed.

In [ ]:
import time

lock_a = threading.Lock()
lock_b = threading.Lock()


def worker_1():
    with lock_a:
        time.sleep(0.1)
        with lock_b:
            print("worker_1 acquired both locks")


def worker_2():
    with lock_b:
        time.sleep(0.1)
        with lock_a:
            print("worker_2 acquired both locks")


t1 = threading.Thread(target=worker_1)
t2 = threading.Thread(target=worker_2)
t1.start()
t2.start()
t1.join(timeout=2)
t2.join(timeout=2)
print("finished (or timed out waiting)")

`worker_1` acquires `lock_a` and then, after a short delay, tries for `lock_b`. `worker_2` acquires `lock_b` and then tries for `lock_a`. If both threads get through their first `with` before either reaches its second, each is left waiting for a lock the other one is holding — permanently, since neither will ever release what it's holding until it acquires the other. The `.join(timeout=2)` calls above exist specifically so this cell doesn't hang the whole notebook forever if a deadlock actually occurs; a real deadlock, without a timeout anywhere, simply never resolves on its own. The standard, reliable defense against this specific shape of deadlock is disciplined: always acquire multiple locks in the same, fixed, agreed-upon order, everywhere in a program — if both workers above always acquired `lock_a` before `lock_b`, the deadlock could never occur, because neither thread would ever be waiting to acquire a lock that a different thread is holding while waiting for the first one's lock in the reverse order.

### `RLock` — a lock a thread can safely acquire more than once

An ordinary `Lock`, acquired a second time by the *same* thread that already holds it, deadlocks against itself — a subtle trap for recursive functions or methods that call each other while holding a lock. `RLock` (a reentrant lock) tracks *which* thread holds it and how many times, allowing that same thread to acquire it again without blocking, as long as it eventually releases it the same number of times.

### `Event` — a simple signal between threads

An `Event` is a flag one thread can set, and any number of other threads can wait on — useful whenever some threads need to pause until a specific condition, set elsewhere, becomes true.

In [ ]:
ready = threading.Event()


def waiter(name):
    print(f"{name}: waiting for the signal")
    ready.wait()
    print(f"{name}: signal received, proceeding")


def setter():
    time.sleep(1)
    print("setter: setting the signal")
    ready.set()


threads = [threading.Thread(target=waiter, args=(f"waiter-{i}",)) for i in range(3)]
setter_thread = threading.Thread(target=setter)

for t in threads:
    t.start()
setter_thread.start()

for t in threads:
    t.join()
setter_thread.join()

All three waiters block on `ready.wait()` until `setter` calls `ready.set()`, at which point every waiting thread is released at once — a clean way to coordinate "don't start until this shared condition is true," without any thread needing to poll a shared variable in a loop.

### `Semaphore` — limiting how many threads can proceed at once

Where a `Lock` allows exactly one thread through at a time, a `Semaphore(n)` allows up to `n` threads through simultaneously — useful for capping concurrent access to a limited resource (a fixed number of database connections, say), without forbidding concurrency entirely.

### Queues and the producer-consumer pattern

A `queue.Queue` is a thread-safe structure specifically designed for passing items between threads — one or more **producer** threads add items, one or more **consumer** threads remove and process them, with the queue itself handling all the necessary internal locking so neither side needs to manage synchronization by hand.

In [ ]:
import queue

work_queue = queue.Queue()


def producer():
    for i in range(5):
        item = f"item-{i}"
        print(f"producing {item}")
        work_queue.put(item)
        time.sleep(0.1)
    work_queue.put(None)   # a sentinel value signaling "no more items"


def consumer():
    while True:
        item = work_queue.get()
        if item is None:
            break
        print(f"consuming {item}")
        time.sleep(0.15)


producer_thread = threading.Thread(target=producer)
consumer_thread = threading.Thread(target=consumer)

producer_thread.start()
consumer_thread.start()

producer_thread.join()
consumer_thread.join()

`work_queue.put()` and `work_queue.get()` are both safe to call from multiple threads simultaneously without any additional locking on your part — `Queue` handles that internally. The `None` sentinel is a common, simple convention for telling a consumer "the producer is finished"; real systems sometimes use a dedicated queue-closing mechanism instead, but the underlying idea — an explicit signal that no more work is coming — is the same.

### One problem

> Write a producer-consumer program with two producer threads adding numbers to a shared `queue.Queue`, and one consumer thread that sums everything it receives, using a `Lock` to protect a shared running-total variable updated from the consumer, and printing the final total once both producers have finished and signaled completion.

In [ ]:
import threading
import queue
import time

# TODO: implement the two-producer, one-consumer program described above


## 80. Choosing Between Threads, Processes, and Async

The last four chapters demonstrated, empirically, that no single concurrency tool is universally correct. This chapter turns those demonstrations into an explicit decision.

### The dimensions that actually matter

**Workload type — CPU-bound or I/O-bound.** This is the single most important question, and Chapters 76 through 78 answered it precisely: CPU-bound work needs genuine parallelism to speed up, which only separate processes provide, because of the GIL. I/O-bound work needs to overlap *waiting*, which both threads and async coroutines can do, since both release control during a wait, just through different mechanisms (preemptive for threads, cooperative for coroutines).

**Shared state.** Threads share memory automatically, which is convenient but requires deliberate synchronization (Chapter 79's locks, events, queues) to stay correct. Processes share nothing automatically, which is safer by default but requires explicit, serialized communication for anything that does need to be shared. `asyncio` coroutines run on a single thread, so — barring genuinely CPU-bound code inside a coroutine blocking the loop, which Chapter 73 warned against — there's no race condition risk at all between coroutines, since only one is ever actually running at a given instant.

**Communication overhead.** Threads communicate essentially for free (shared memory, no copying). Processes pay a real serialization cost for every piece of data crossing the boundary (Chapter 77). Async coroutines, being in the same process and even the same thread, communicate as cheaply as threads do.

**Scale and complexity.** Starting a thread is cheap; starting a process is considerably more expensive (a whole new interpreter). `asyncio` can comfortably manage thousands of concurrent coroutines waiting on network operations — far more than would be practical as separate OS threads, each of which carries real memory and scheduling overhead.

### A decision framework, stated directly

```text
Is the work CPU-bound (heavy computation, little or no waiting)?
    │
    YES → use multiprocessing.
    │      Threads and async provide no real speedup here (GIL / single thread).
    │
    NO → the work is I/O-bound (waiting on network, disk, or similar)
            │
            Does the codebase, and the libraries involved,
            already support async (e.g. an async database driver,
            async HTTP client)?
                │
                YES → prefer asyncio.
                │      Lower overhead than threads at scale; no
                │      shared-memory race conditions to manage.
                │
                NO  → threading is a reasonable, simpler choice,
                       particularly for a small number of concurrent
                       operations or when integrating with
                       synchronous-only libraries.
```

### Working through realistic scenarios

**A web scraper fetching a thousand pages.** I/O-bound — each request spends nearly all its time waiting for a response, not computing. This is close to the canonical `asyncio` use case: with an async HTTP client, a single thread can have hundreds of requests in flight simultaneously, at a fraction of the memory and scheduling cost of a thousand OS threads.

**Resizing ten thousand images.** CPU-bound — decoding, resizing, and re-encoding an image is genuine computation with essentially no waiting involved. `multiprocessing`, distributing images across a pool sized to the number of available CPU cores, is the correct choice; threads would provide no speedup at all here, for exactly the reason Chapter 76 demonstrated directly.

**A GUI application that needs to stay responsive while running a slow background calculation.** This is a case worth pausing on, because "responsive" sounds like an I/O question but the calculation itself is CPU-bound. Running the calculation on a background *thread* is still often the pragmatic choice here — not because it makes the calculation faster (it won't, per Chapter 78), but because it prevents that calculation from blocking the thread responsible for keeping the interface responsive. This is a legitimate use of threading for a CPU-bound task, but notice its actual purpose: keeping one thread free to do something else, not speeding up the computation itself. A CPU-bound background calculation that needs to be genuinely *faster*, not merely non-blocking, still calls for `multiprocessing`.

**A server handling many simultaneous client connections, each doing modest computation between reads.** A mixed workload — mostly I/O-bound (waiting for client data) with some CPU-bound work interleaved. `asyncio` handles the waiting efficiently; the CPU-bound portions, if substantial, may need to be offloaded to a process pool from within the async program (a hybrid approach — `asyncio` provides exactly this via `loop.run_in_executor`, worth knowing exists even without full coverage here) so they don't block the event loop the way Chapter 73 warned a long CPU-bound stretch inside a coroutine would.

### The trade-off underneath all of it

Every one of these tools trades some form of complexity for some form of overlap. Threads trade synchronization discipline (Chapter 79) for cheap, shared-memory concurrency. Processes trade communication cost and startup overhead for genuine parallelism. Async trades a somewhat different programming model (`async`/`await`, threading through your whole call stack once adopted) for extremely low-overhead concurrency at scale. None of them is a default to reach for out of habit; each is a specific answer to a specific shape of workload, and the honest first question, every time, is the one Chapter 76 opened with: is this work waiting, or is it computing?

### One problem

> For each of the following three scenarios, state which concurrency model (synchronous, threading, multiprocessing, or asyncio) you would choose and justify it in two or three sentences referencing this chapter's dimensions: (1) computing SHA-256 hashes of ten thousand large files already on disk; (2) sending push notifications to fifty thousand mobile devices over the network; (3) running a physics simulation that updates a shared grid of values every timestep, where each cell's update depends on its neighbors' previous values.

In [ ]:
# TODO: write your reasoning for each of the three scenarios as comments


# Part V — Performance and Memory

## 81. Python Memory Model and Object Lifetime

Chapter 61 established `is` as an identity check and briefly used `id()` to show it. This chapter builds a complete, precise picture of what an object actually is in memory, and what happens to one when nothing refers to it anymore.

### Every value is an object, with an identity, a type, and a value

```text
id(obj)      a number identifying this specific object (CPython: its memory address)
type(obj)    what kind of object it is -- fixed for its entire lifetime
obj itself   its current value/state -- can change, if the type is mutable
```

`id()` and `type()` never change for a given object over its lifetime, no matter what happens to its value: 

In [ ]:
x = [1, 2, 3]
print(id(x), type(x))

x.append(4)
print(id(x), type(x))   # same id, same type -- the object itself was mutated

x = [9, 9, 9]
print(id(x), type(x))   # different id -- this is now a genuinely different object

`x.append(4)` mutated the existing list in place -- same `id()`, same object, just a changed internal state. `x = [9, 9, 9]` rebound the name `x` to an entirely new object -- a different `id()`. This is the exact distinction Basics' Chapter 4 introduced informally with the "redirecting an arrow versus editing what it points to" picture; the data model simply gives it a precise, checkable form.

### Reference counting, conceptually

CPython tracks, for every object, how many references currently point to it -- a **reference count**. Every time a name is bound to an object, or the object is stored in a container, or passed as an argument, its count goes up; every time a reference goes away (a name is reassigned, a local variable's function returns, a container is cleared), the count goes down. When an object's reference count reaches zero -- nothing anywhere in the program refers to it anymore -- CPython deallocates it immediately.

In [ ]:
import sys

x = [1, 2, 3]
print(sys.getrefcount(x))

y = x
print(sys.getrefcount(x))   # one more reference now exists (y)

del y
print(sys.getrefcount(x))   # back down

Note that `sys.getrefcount` always reports one more than you might expect, because passing `x` into `getrefcount` itself creates one additional, temporary reference for the duration of the call. This is worth flagging precisely as a **CPython implementation detail**, not a guarantee of the Python language: reference counting is how CPython specifically manages memory, and `sys.getrefcount` only exists, and only reports something meaningful, because of that specific implementation choice. Other Python implementations are free to manage memory entirely differently, while still being fully valid implementations of the Python *language* -- object lifetime and cleanup are not part of what the language itself formally promises, only part of what CPython, specifically, happens to do.

### Aliasing, revisited with full precision

Two names alias each other when they hold the same `id()`, meaning both are, in reference-counting terms, contributing to the *same* object's reference count.

In [ ]:
a = [1, 2, 3]
b = a

print(a is b)
print(id(a) == id(b))

b.append(4)
print(a)

Intermediate Chapter 34's warning about two names sharing a mutated list is now visible as a direct consequence of identity: `a` and `b` are two names for one object, so any mutation through either name is visible through the other, because there was never a second object to keep separate in the first place.

### Shallow copy versus deep copy

Basics and Intermediate both used `.copy()` on a list to get an independent object. It's worth being precise about exactly how independent that copy actually is, because the answer changes once a list contains other mutable objects.

In [ ]:
original = [[1, 2], [3, 4]]
shallow = original.copy()

shallow.append([5, 6])
print(original)
print(shallow)

shallow[0].append(99)
print(original)
print(shallow)

`shallow.append([5, 6])` only affected `shallow` -- appending to the outer list doesn't touch `original` at all, since `.copy()` did create a genuinely separate outer list. But `shallow[0].append(99)` affected **both** `original` and `shallow`. The reason: `.copy()` performs a **shallow copy** -- it creates a new outer list, but the *elements* inside that new list are the same objects as the elements inside the original, not copies of them. `original[0]` and `shallow[0]` are the same inner list, referenced from two different outer lists.

```text
original ──► [ ref, ref ]        shallow ──► [ ref, ref ]
                │    │                          │    │
                └────┼──────────────────────────┘    │
                     └───────────────────────────────┘
              (both outer lists' first two elements point to the SAME two inner lists)
```

A **deep copy**, from the `copy` module, recursively copies every nested mutable object too, producing something genuinely, fully independent all the way down.

In [ ]:
import copy

original = [[1, 2], [3, 4]]
deep = copy.deepcopy(original)

deep[0].append(99)
print(original)
print(deep)

Now mutating `deep`'s nested list has no effect on `original`'s -- every level of the structure was copied, not merely the outermost container. The rule worth carrying forward: `.copy()` (or `list(x)`, or a slice `x[:]`) is sufficient exactly when a container holds only immutable elements, or when sharing the nested objects is actually intended; the moment a container holds mutable elements that must be genuinely independent, `copy.deepcopy` is the correct tool, at the cost of copying considerably more data.

### One problem

> Write a function `independent_copy(matrix)` that takes a list of lists (representing a grid) and returns a fully independent deep copy, without using `copy.deepcopy` directly -- build it manually using a nested comprehension, then verify its independence by mutating a cell in the copy and confirming the original is unaffected.

In [ ]:
# TODO: implement independent_copy(matrix) using a nested comprehension


## 82. Garbage Collection and Reference Cycles

Reference counting, as described in the last chapter, sounds like a complete memory-management story: an object disappears the instant nothing refers to it anymore. It is not complete, and the gap is worth understanding precisely, because it's the direct answer to how a program written entirely in a "memory-managed" language can still leak memory.

### The gap: reference cycles

Consider two objects that end up referring to each other.

In [ ]:
class Node:
    def __init__(self, name):
        self.name = name
        self.partner = None


a = Node("A")
b = Node("B")

a.partner = b
b.partner = a

del a
del b

# At this point, nothing named in this program refers to either Node --
# but each Node still refers to the other, through .partner.

After `del a` and `del b`, no *name* in the program refers to either `Node` object anymore. But `a`'s old object still has its `.partner` pointing at `b`'s old object, and vice versa -- each one still holds a reference to the other, keeping both of their reference counts above zero, even though the program itself has no way left to reach either of them.

```text
   ┌─────────────┐        ┌─────────────┐
   │   Node A    │──────► │   Node B    │
   │  .partner ──┼───┐    │  .partner ──┼───┐
   └─────────────┘   │    └─────────────┘   │
         ▲           └──────────────────────┘
         └───────────────────────────────────┘
   (each keeps the other's reference count above zero -- neither reaches zero)
```

This is a **reference cycle**: a group of objects referencing each other in a loop, with no reference reaching them from outside the loop. Pure reference counting, as described in the last chapter, can never collect these -- each object's count never drops to zero, because the cycle itself is still holding on to both objects, permanently, even though the rest of the program has completely lost access to them.

### The cyclic garbage collector

CPython includes a second, separate mechanism specifically to catch this gap: the **cyclic garbage collector**, in the `gc` module. Periodically (and configurably), it looks for groups of objects that reference each other but aren't reachable from anywhere else in the program -- exactly the situation above -- and reclaims them, even though their individual reference counts never reached zero on their own.

In [ ]:
import gc

class Node:
    def __init__(self, name):
        self.name = name
        self.partner = None
    def __del__(self):
        print(f"{self.name} is being destroyed")


a = Node("A")
b = Node("B")
a.partner = b
b.partner = a

del a
del b

print("cycle created and names deleted -- objects not yet necessarily collected")
gc.collect()
print("after gc.collect()")

`__del__` is a dunder method called when an object is actually about to be destroyed -- here used purely to make destruction visible. Depending on timing, the two `Node` objects may or may not already be gone by the time `gc.collect()` runs (CPython's cyclic collector does run automatically, on its own schedule, not only when called explicitly) -- but the explicit call guarantees a collection pass happens at that point, and the destruction messages become visible as a direct result of the cycle finally being broken.

### Why this still counts as a memory leak risk

None of this is automatic in the specific sense reference counting is. The cyclic collector runs periodically, based on internal thresholds tracking how many objects have been allocated since the last pass -- not the instant a cycle becomes unreachable. In a program that creates many short-lived cyclic structures rapidly, memory usage can grow noticeably before a collection pass catches up, which makes the collector's timing genuinely relevant to real performance -- and in some cases -- particularly objects that define `__del__` in more complex ways, or extension modules holding references outside Python's own visibility -- cycles can resist collection entirely. "Python manages memory automatically" is a true statement about the common case and a dangerously incomplete one for diagnosing an actual leak; understanding that cycles are a distinct mechanism, with distinct timing, from reference counting is what makes that diagnosis possible at all.

### `weakref` -- deliberately not counting as a reference

Sometimes two objects need to refer to each other by design (a parent-child relationship, a cache pointing back to what it caches) without that reference keeping either one alive artificially. A **weak reference**, from the `weakref` module, refers to an object without incrementing its reference count at all.

In [ ]:
import weakref

class Node:
    def __init__(self, name):
        self.name = name


a = Node("A")
weak_ref_to_a = weakref.ref(a)

print(weak_ref_to_a())

del a
print(weak_ref_to_a())

`weak_ref_to_a` does not keep `a`'s object alive -- once the only *strong* reference (the name `a` itself) is deleted, the object is freed immediately, exactly as reference counting would normally guarantee, and calling `weak_ref_to_a()` afterward correctly returns `None` rather than a now-dangling object. Using weak references for exactly the kind of back-reference the `Node` example built earlier would have avoided the cycle from ever forming in the first place -- `.partner` as a weak reference, rather than an ordinary one, means neither `Node` keeps the other alive artificially, and ordinary reference counting alone becomes sufficient to reclaim both the moment nothing else refers to them.

### One problem

> Build a small parent-child class structure where each parent holds a list of children, and each child holds a reference back to its parent -- first using an ordinary reference (creating a cycle), then rewritten using `weakref.ref` for the child's back-reference to its parent. Add a `__del__` method to observe destruction timing, and use `gc.collect()` to demonstrate the difference between the two versions.

In [ ]:
import weakref
import gc

# TODO: build both versions of the parent-child structure and compare destruction timing


## 83. Measuring Performance with `timeit` and Profiling

Every optimization this chapter and the next one discuss depends entirely on one discipline, stated as plainly as possible: **never optimize based on a guess about what's slow.** Guesses about performance are wrong often enough, even among experienced developers, that measurement is not an optional nicety -- it's the only reliable way to know where a program's time is actually going.

### `timeit` -- measuring a small piece of code reliably

A single `time.perf_counter()` measurement, as used in earlier chapters of this book, is enough to compare two clearly different approaches, but it's noisy for anything subtle -- background system activity, one-off delays, and general timing jitter can easily swamp a small, genuine difference. `timeit` runs a snippet many times and reports a much more stable result.

In [ ]:
import timeit

def using_loop():
    result = []
    for i in range(1000):
        result.append(i * i)
    return result


def using_comprehension():
    return [i * i for i in range(1000)]


loop_time = timeit.timeit(using_loop, number=1000)
comprehension_time = timeit.timeit(using_comprehension, number=1000)

print(f"loop:          {loop_time:.4f} seconds total")
print(f"comprehension: {comprehension_time:.4f} seconds total")

`number=1000` runs each function a thousand times and reports the *total* time across all runs, specifically to average out timing noise that would dominate a single execution of something this fast. This is the honest way to compare two small alternatives -- Intermediate Chapter 31's claim that comprehensions are often faster than the equivalent loop is exactly the kind of claim that should be checked this way, rather than accepted on reputation alone.

### Why a single measurement misleads

Run the same comparison more than once, and the two numbers will shift somewhat from run to run -- other processes on the machine, memory allocation patterns, and CPU frequency scaling all introduce real variance. A single `timeit` call already helps by repeating the measurement internally, but for anything performance-critical, running the whole comparison multiple times and looking at the *range* of results, not just one number, is worth the extra effort; a single measurement that happens to be unusually fast or slow can lead directly to a wrong conclusion about which approach is actually better.

### `cProfile` -- finding out where a whole program's time actually goes

`timeit` is for comparing small, isolated snippets. `cProfile` answers a different, larger question: for an entire function (or program), which of the many function calls inside it are actually consuming the time.

In [ ]:
import cProfile
import pstats
import io


def slow_step():
    total = 0
    for i in range(500_000):
        total += i
    return total


def fast_step():
    return sum(range(500_000))


def main():
    for _ in range(3):
        slow_step()
    for _ in range(10):
        fast_step()


profiler = cProfile.Profile()
profiler.enable()
main()
profiler.disable()

stream = io.StringIO()
stats = pstats.Stats(profiler, stream=stream).sort_stats("cumulative")
stats.print_stats(5)
print(stream.getvalue())

The output reports, for every function actually called during `main()`, how many times it was called and how much time was spent inside it -- both including (`cumulative`) and excluding (`tottime`) time spent in functions it called. Sorted by cumulative time, `slow_step` should stand out clearly as consuming far more of the total, despite being called fewer times than `fast_step` -- exactly the kind of finding that turns "I think this part is slow" into "this part is measurably, specifically slow, and here is by how much."

### Interpreting results honestly

A profiler answers "where does the time go," not "what should I change." The two most common misreadings are worth naming directly: treating a function with a high call count as automatically a good optimization target, when the *per-call* cost might be negligible and the real cost lies elsewhere; and optimizing the function the profiler happens to display first, rather than the one actually contributing the largest share of total time. Sorting by cumulative time and looking specifically at where the largest *fraction* of total execution time concentrates is the right way to identify a genuine bottleneck, rather than reacting to whichever number is visually largest or most surprising.

### The discipline, stated as a loop

```text
Measure  →  Identify the actual bottleneck  →  Change one thing  →  Measure again
    ▲                                                                    │
    └────────────────────────────────────────────────────────────────────┘
```

Skipping the final "measure again" step is a common mistake in its own right -- a change made in response to profiling data is a hypothesis about what will help, not a guarantee, and confirming it actually helped (and didn't quietly make something else worse) closes the loop honestly, rather than trusting that a change *should* have worked.

### One problem

> Write two implementations of a function that checks whether a number is prime -- one using a naive loop up to `n`, one using a loop up to `sqrt(n)` -- and use `timeit` to measure and compare their performance for checking primality of a list of twenty large numbers, printing both timings.

In [ ]:
import timeit
import math

# TODO: implement both prime-checking functions and compare their timing


## 84. Optimization -- Finding the Real Bottleneck

The last chapter established measurement as a discipline. This chapter is about what to actually do once measurement has identified a genuine bottleneck -- and, just as importantly, about the mistakes that come from optimizing without that discipline.

### Algorithmic complexity usually dominates everything else

Before considering any lower-level optimization, the shape of the algorithm itself is almost always the largest factor in how a program's performance scales.

In [ ]:
import timeit


def has_duplicate_slow(items):
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            if items[i] == items[j]:
                return True
    return False


def has_duplicate_fast(items):
    seen = set()
    for item in items:
        if item in seen:
            return True
        seen.add(item)
    return False


data = list(range(2000)) + [500]   # a duplicate hidden near the end

slow_time = timeit.timeit(lambda: has_duplicate_slow(data), number=10)
fast_time = timeit.timeit(lambda: has_duplicate_fast(data), number=10)

print(f"nested loop (roughly n^2): {slow_time:.4f} seconds")
print(f"set-based (roughly n):     {fast_time:.4f} seconds")

The gap here is not a matter of micro-level tuning -- it's a difference in **algorithmic complexity**: the nested-loop version does roughly `n^2` comparisons in the worst case, while the set-based version does roughly `n`, because checking membership in a set (Basics Chapter 20) is a fast operation regardless of the set's size, unlike checking membership in a list by scanning it. As `data` grows larger, this gap widens dramatically -- doubling the input size roughly quadruples the slow version's work but only roughly doubles the fast version's. No amount of micro-optimizing the inner loop of `has_duplicate_slow` would ever close a gap this fundamental; the fix is a different algorithm, not a faster version of the same one.

### Repeated computation and caching

A function recomputing the same result multiple times, when the input hasn't changed, is a common and easily fixed source of wasted work -- exactly the situation Intermediate Chapter 49's `lru_cache` was built for.

In [ ]:
def fibonacci_uncached(n):
    if n < 2:
        return n
    return fibonacci_uncached(n - 1) + fibonacci_uncached(n - 2)


from functools import lru_cache


@lru_cache
def fibonacci_cached(n):
    if n < 2:
        return n
    return fibonacci_cached(n - 1) + fibonacci_cached(n - 2)


uncached_time = timeit.timeit(lambda: fibonacci_uncached(25), number=5)
cached_time = timeit.timeit(lambda: fibonacci_cached(25), number=5)

print(f"uncached: {uncached_time:.4f} seconds")
print(f"cached:   {cached_time:.6f} seconds")

The cached version wins by orders of magnitude, because the uncached version recomputes the same smaller Fibonacci values an exponential number of times, while the cached version computes each distinct input exactly once and reuses the stored result for every subsequent request. This is worth recognizing as its own category of bottleneck, separate from algorithmic complexity in the abstract sense: the *algorithm* here has an exponential blow-up specifically because of redundant recomputation, and caching removes the redundancy directly rather than requiring a fundamentally different approach.

### The right data structure for the operation you actually perform most

The `has_duplicate` example above is really a special case of a broader principle: choosing a data structure based on what operation the code performs most frequently, not out of habit. Checking membership repeatedly favors a `set` or `dict` over a `list`; needing to add and remove from both ends efficiently favors a `deque` (Intermediate Chapter 57) over a `list`; needing fast lookup by a key favors a `dict` over scanning parallel lists. This is frequently a bigger win than any lower-level tuning, and it's worth checking before reaching for a profiler at all, precisely because it's often visible from the code's shape alone.

### Vectorization, conceptually

For genuinely numeric, bulk operations -- applying the same arithmetic across a large array of numbers -- specialized libraries like NumPy perform the operation using compiled, low-level code operating on contiguous blocks of memory, rather than looping over each element with ordinary Python bytecode, one at a time. This is called **vectorization**, and the performance difference for large numeric workloads can be dramatic -- often one or two orders of magnitude. This book doesn't teach NumPy directly (that belongs to a dedicated data-science curriculum), but the concept is worth knowing by name: if a workload is fundamentally "the same arithmetic operation, repeated over a large, uniform collection of numbers," a hand-written Python loop is very often not the right tool at all, regardless of how well that loop itself is optimized.

### Readability versus performance -- a trade-off, not a contest

Not every optimization is worth making. A change that makes code meaningfully harder to read, in exchange for a speedup that has no measurable effect on the program's actual behavior (because that code path runs rarely, or the absolute time involved is negligible), is a net loss. The measurement discipline from the last chapter is precisely what prevents this mistake: without profiling data showing that a specific piece of code is actually a meaningful fraction of total run time, "optimizing" it is, at best, a wash, and at worst, actively harmful to the code's maintainability for no real benefit. Premature optimization -- restructuring code for speed before measurement has shown it's needed -- is worth avoiding specifically because it trades a certain cost (worse readability, more complexity) for an uncertain, frequently nonexistent benefit.

### One problem

> Given a large list of dictionaries representing orders (each with a `customer_id` field), write a function that finds all orders belonging to a specific customer. Implement it two ways -- scanning the list each time, and building a `dict` mapping `customer_id` to a list of that customer's orders once, then looking up by key -- and use `timeit` to compare their performance across ten separate lookups.

In [ ]:
import timeit
import random

orders = [{"customer_id": random.randint(1, 1000), "amount": random.randint(10, 500)} for _ in range(20000)]

# TODO: implement both lookup strategies and compare their timing across ten lookups


## 85. Efficient Data Processing and Memory Usage

Chapter 84 focused on speed. This chapter focuses on the related, sometimes competing concern of memory -- how much of it a program actually needs at once, and how the tools already covered across all three volumes can be used deliberately to keep that footprint small.

### The cost of materializing a full collection

Intermediate Chapters 45 and 46 already demonstrated the core idea: a generator computes values lazily, one at a time, while a list computes and holds everything at once. This chapter's contribution is treating that choice as a deliberate memory-management decision, not just a stylistic one.

In [ ]:
import sys

list_version = [x * x for x in range(1_000_000)]
generator_version = (x * x for x in range(1_000_000))

print(f"list:      {sys.getsizeof(list_version):,} bytes")
print(f"generator: {sys.getsizeof(generator_version):,} bytes")

The list's reported size reflects roughly a million stored integer references; the generator's reported size is small and constant, regardless of how many values it will eventually produce, because it isn't storing any of them -- only the state needed to compute the next one on demand. For a workload that only ever needs to look at each value once, in order (summing, searching, filtering into a smaller result), a generator accomplishes the task using a small, constant amount of memory rather than an amount that grows directly with the input size.

### Streaming a large file instead of loading it whole

This is the single most common, practical place this idea matters. Compare reading an entire file into memory before processing it against processing it line by line.

In [ ]:
with open("large_sample.txt", "w") as f:
    for i in range(200_000):
        f.write(f"line {i}: some sample data here\n")

In [ ]:
def count_matches_load_all(filename, keyword):
    with open(filename, "r") as f:
        lines = f.readlines()   # the whole file, all at once, in memory
    return sum(1 for line in lines if keyword in line)


def count_matches_streaming(filename, keyword):
    count = 0
    with open(filename, "r") as f:
        for line in f:          # one line at a time -- Basics Chapter 26's iteration over a file
            if keyword in line:
                count += 1
    return count


print(count_matches_load_all("large_sample.txt", "9999"))
print(count_matches_streaming("large_sample.txt", "9999"))

Both functions produce the same count, but `count_matches_load_all` holds every line of the file in memory simultaneously via `.readlines()`, while `count_matches_streaming` never holds more than one line at a time -- a file too large to comfortably fit in memory at once is processable by the second version and not, practically, by the first. This is precisely the file-iteration behavior Basics Chapter 26 introduced (`for line in file:`) revisited here as a deliberate memory strategy rather than just a syntax convenience: the file object itself is an iterator, producing lines lazily, exactly like the generators from Intermediate Chapters 45 and 46.

### Batching -- a middle ground between one-at-a-time and all-at-once

Sometimes processing strictly one item at a time is inefficient for a different reason -- a database insert, or a network call, has a fixed overhead cost per call, and doing that a million separate times is wasteful even if each individual item is small. **Batching** processes a fixed-size chunk at a time, trading a small, bounded amount of memory for a large reduction in per-item overhead.

In [ ]:
def batched(iterable, batch_size):
    batch = []
    for item in iterable:
        batch.append(item)
        if len(batch) == batch_size:
            yield batch
            batch = []
    if batch:
        yield batch


def process_batch(batch):
    return sum(batch)


total = 0
for batch in batched(range(1_000_000), batch_size=10_000):
    total += process_batch(batch)

print(total)

`batched` is itself a generator -- it never holds more than `batch_size` items in memory at once, while still allowing whatever processes each batch to work on a reasonably sized chunk rather than a single item, striking a deliberate balance between the memory profile of pure streaming and the overhead-per-call cost of handling items strictly one at a time. (Python 3.12 added `itertools.batched` as a standard-library version of exactly this pattern, worth reaching for directly once available rather than rewriting it by hand.)

### Avoiding unnecessary intermediate copies

A chain of transformations written as successive list comprehensions creates a full intermediate list at every single step, even when only the final result is ever used.

In [ ]:
numbers = list(range(1_000_000))

# Three full intermediate lists are built and briefly held in memory here.
step1 = [n * 2 for n in numbers]
step2 = [n for n in step1 if n % 3 == 0]
step3 = [n ** 0.5 for n in step2]
result_eager = sum(step3)

# A single generator pipeline (Chapter 71) builds none of these intermediate lists --
# each value flows through all three transformations before the next one is even produced.
doubled = (n * 2 for n in numbers)
filtered = (n for n in doubled if n % 3 == 0)
rooted = (n ** 0.5 for n in filtered)
result_lazy = sum(rooted)

print(result_eager)
print(result_lazy)

Both produce the same total, but the eager version briefly holds three separate million-element lists in memory (`step1`, `step2`, and `step3`, before each is discarded), while the lazy version holds none of them -- `doubled`, `filtered`, and `rooted` are all generator objects, and `sum` pulls exactly one final value at a time, driving all three stages together for that one value, precisely the pipeline mechanism traced in full in Chapter 71. This is the same trade-off as always: the eager version is arguably easier to inspect step by step while debugging (you can print `step2` directly and look at it), while the lazy version uses meaningfully less peak memory for a large input, at the cost of not having any single intermediate collection to inspect after the fact.

### Relating this to data-heavy workflows without becoming a different course

Real data-processing work -- the kind done with tools like Pandas or NumPy -- runs into exactly these same considerations at a larger scale: loading an entire dataset into memory at once works fine until the dataset stops comfortably fitting in available memory, at which point streaming, chunked, or batched processing (all covered in this chapter, using nothing beyond generators and ordinary Python) become necessary rather than optional. This book doesn't teach those specialized libraries, but the underlying discipline -- know whether your data needs to exist all at once, or whether it can flow through your program one piece at a time -- transfers directly, and is worth having internalized before ever reaching for a library that hides these choices behind a more convenient interface.

### One problem

> Write a function `process_large_file(filename, chunk_size)` that reads a large text file in fixed-size batches of lines (using the `batched` generator from this chapter, or an equivalent), computing and printing the average line length for each batch, without ever loading the whole file into memory at once.

In [ ]:
# TODO: define process_large_file(filename, chunk_size) using batched streaming


# Part VI — Professional Python Engineering

## 86. Logging, Configuration, and Environment Management

Every example in this book, across all three volumes, has used `print()` to show what's happening. That's the right choice for a notebook meant to be read. It's the wrong choice for a real, running program, and it's worth being specific about exactly why, rather than treating this as an arbitrary rule.

### Why `print()` doesn't scale to a real application

`print()` always writes to standard output, always as plain text, with no record of when it happened, how severe it was, or which part of the program it came from. Once a program runs unattended -- a server, a scheduled job, a background service -- nobody is watching its output live; whatever it wrote needs to be findable later, filterable by severity, and traceable to its source, none of which `print()` provides on its own.

### `logging` -- severity levels

The standard library's `logging` module organizes messages by **level**, in increasing order of severity: `DEBUG`, `INFO`, `WARNING`, `ERROR`, `CRITICAL`.

In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

logger = logging.getLogger(__name__)

logger.debug("Detailed diagnostic info -- not shown at INFO level")
logger.info("Application started")
logger.warning("Configuration value missing, using default")
logger.error("Failed to connect to database")

`logging.basicConfig(level=logging.INFO, ...)` sets the minimum level that actually gets output -- `logger.debug(...)` produced nothing here, because `DEBUG` sits below the configured `INFO` threshold. This is the single most immediately useful thing logging offers over `print()`: the same code, unchanged, can run verbosely during development (`level=logging.DEBUG`) and quietly in production (`level=logging.WARNING`), controlled entirely by configuration rather than by editing or commenting out `print()` calls throughout the codebase.

`getLogger(__name__)` is a near-universal convention worth adopting directly: it names the logger after the current module, so log output can be traced back to exactly where it originated, which matters considerably once a program spans more than one file.

### Handlers and formatters -- where messages go, and how they look

A **handler** determines where a log message is sent -- the console, a file, a network service. A **formatter** determines how each message is rendered as text. `basicConfig` above configured both implicitly; here they're set up explicitly, which is what a real application typically needs, since it usually wants log output in more than one place.

In [ ]:
import logging

logger = logging.getLogger("myapp")
logger.setLevel(logging.DEBUG)
logger.handlers.clear()

file_handler = logging.FileHandler("app.log")
file_handler.setLevel(logging.DEBUG)

console_handler = logging.StreamHandler()
console_handler.setLevel(logging.WARNING)

formatter = logging.Formatter("%(asctime)s [%(levelname)s] %(name)s: %(message)s")
file_handler.setFormatter(formatter)
console_handler.setFormatter(formatter)

logger.addHandler(file_handler)
logger.addHandler(console_handler)

logger.info("This goes to the file, but not the console")
logger.error("This goes to both")

with open("app.log") as f:
    print(f.read())

The file handler captures everything from `DEBUG` upward; the console handler only shows `WARNING` and above -- two independent thresholds, applied to the same underlying log calls, letting a detailed record accumulate in a file while only the genuinely important messages interrupt whoever is watching the console live.

### Logging an exception properly

Intermediate Chapter 52 taught catching exceptions deliberately rather than swallowing them silently. `logger.exception(...)`, called from inside an `except` block, records the message *and* the full traceback -- the specific tool for making a caught failure visible in the log record, rather than only in whatever ad hoc `print(e)` a developer might otherwise reach for.

In [ ]:
try:
    result = 10 / 0
except ZeroDivisionError:
    logger.exception("Division failed")

with open("app.log") as f:
    print(f.read())

### Separating configuration from code

A program's behavior frequently needs to vary between environments -- a development database versus a production one, a debug logging level locally versus a quiet one in production -- without editing the source code itself for each environment. **Environment variables**, read via `os.environ`, are the standard mechanism for exactly this.

In [ ]:
import os

database_url = os.environ.get("DATABASE_URL", "sqlite:///default.db")
debug_mode = os.environ.get("DEBUG", "false").lower() == "true"

print(database_url)
print(debug_mode)

`os.environ.get(..., default)` mirrors Basics Chapter 19's dictionary `.get()` exactly, because `os.environ` genuinely behaves like a dictionary of the current process's environment variables. This is worth using specifically for values that legitimately differ between environments and, critically, for **secrets** -- API keys, database passwords, and similar sensitive values that should never be written directly into source code (and therefore never committed to version control alongside it). A configuration value hardcoded in a file is visible to anyone who can read that file's history; a value read from the environment is supplied separately, at deploy time, by whoever controls that environment specifically.

### One problem

> Write a small module that configures a logger with both a file handler (capturing `DEBUG` and above) and a console handler (capturing only `WARNING` and above), reads a `LOG_LEVEL` environment variable to optionally override the console handler's level, and demonstrates logging at several different severities, including one caught exception logged with `logger.exception(...)`.

In [ ]:
import logging
import os

# TODO: configure the logger as described and demonstrate its behavior


## 87. Command-Line Applications with `argparse`

Every program in this book has either run its entire logic unconditionally or asked for input interactively via `input()`. Real command-line tools are typically configured through **arguments** supplied when the program is invoked -- `myprogram --verbose input.txt`, rather than a sequence of interactive prompts.

### Reading `sys.argv` directly, and why it's not enough

`sys.argv` is the raw list of command-line arguments a script was invoked with, with the script's own name as the first element.

In [ ]:
script = '''
import sys

print(sys.argv)
'''

with open("raw_args.py", "w") as f:
    f.write(script)

In [ ]:
import subprocess

result = subprocess.run(
    ["python", "raw_args.py", "input.txt", "--verbose"],
    capture_output=True, text=True
)
print(result.stdout)

`sys.argv` works, but everything is a plain, unlabeled string -- there's no automatic distinction between a flag and a value, no type conversion, no validation, and no generated help text. Every one of those would need to be built by hand, and `argparse` exists specifically to provide all of it.

### Positional and optional arguments

**Positional** arguments are required and identified by their position; **optional** arguments (conventionally prefixed with `--`) are identified by name and are typically, though not always, optional in the literal sense.

In [ ]:
script = '''
import argparse

parser = argparse.ArgumentParser(description="A small file-processing tool")
parser.add_argument("filename", help="the file to process")
parser.add_argument("--verbose", action="store_true", help="print detailed output")
parser.add_argument("--limit", type=int, default=10, help="maximum number of lines to process")

args = parser.parse_args()

print(f"filename: {args.filename}")
print(f"verbose: {args.verbose}")
print(f"limit: {args.limit}")
'''

with open("cli_tool.py", "w") as f:
    f.write(script)

In [ ]:
result = subprocess.run(
    ["python", "cli_tool.py", "data.csv", "--verbose", "--limit", "50"],
    capture_output=True, text=True
)
print(result.stdout)

`filename` is positional -- the tool won't run without it, and its meaning is determined purely by where it appears. `--verbose` uses `action="store_true"`, meaning its mere presence sets `args.verbose` to `True`, with no value needed after it (a **flag**, in the terminology this chapter's curriculum names directly). `--limit` expects a value, is automatically converted to an `int` via `type=int` (raising a clear error automatically if given something that isn't a valid integer -- validation `sys.argv` never provided), and falls back to `10` if omitted entirely.

### Generated help, for free

Notice `parser.add_argument(..., help="...")` above -- every one of those descriptions feeds directly into an automatically generated `--help` output, without any additional work.

In [ ]:
result = subprocess.run(["python", "cli_tool.py", "--help"], capture_output=True, text=True)
print(result.stdout)

This is a genuine, practical advantage over hand-parsing `sys.argv`: the help text is generated directly from the same argument definitions the parser actually uses, so it can never silently drift out of sync with what the program actually accepts, the way a hand-written usage string easily could.

### Validation and error messages, also for free

Omitting a required positional argument, or supplying an invalid value for a typed one, produces a specific, useful error message automatically, along with a non-zero exit code signaling failure to whatever invoked the program.

In [ ]:
result = subprocess.run(["python", "cli_tool.py", "--limit", "not-a-number"], capture_output=True, text=True)
print(result.stderr)
print("exit code:", result.returncode)

### Subcommands -- one tool, several distinct operations

Many real command-line tools (`git`, most notably) offer several distinct operations under one program name -- `git commit`, `git push` -- each with its own set of arguments. `argparse` supports this through **subparsers**.

In [ ]:
script = '''
import argparse

parser = argparse.ArgumentParser(description="A small task manager")
subparsers = parser.add_subparsers(dest="command", required=True)

add_parser = subparsers.add_parser("add", help="add a new task")
add_parser.add_argument("title")

list_parser = subparsers.add_parser("list", help="list all tasks")
list_parser.add_argument("--completed-only", action="store_true")

args = parser.parse_args()

if args.command == "add":
    print(f"Adding task: {args.title}")
elif args.command == "list":
    print(f"Listing tasks (completed only: {args.completed_only})")
'''

with open("task_cli.py", "w") as f:
    f.write(script)

In [ ]:
result_add = subprocess.run(["python", "task_cli.py", "add", "Write chapter 87"], capture_output=True, text=True)
print(result_add.stdout)

result_list = subprocess.run(["python", "task_cli.py", "list", "--completed-only"], capture_output=True, text=True)
print(result_list.stdout)

`args.command` reports which subcommand was actually invoked, and each subparser (`add_parser`, `list_parser`) defines only the arguments relevant to its own operation -- `add`'s `title` and `list`'s `--completed-only` are entirely independent, which keeps each subcommand's interface focused, exactly the separation-of-concerns instinct that Chapter 89 will apply more broadly to whole applications.

### One problem

> Build a small CLI tool with two subcommands: `convert`, taking a positional `filename` and an optional `--to` argument (defaulting to `"json"`) specifying the target format, and `validate`, taking a positional `filename` and a `--strict` flag. Write it to disk as a script and demonstrate invoking both subcommands via `subprocess.run`, including one call to `--help`.

In [ ]:
import subprocess

# TODO: write the CLI script to disk and demonstrate both subcommands plus --help


## 88. Dependency Management, Virtual Environments, and Reproducible Projects

Every chapter so far in this book has used either the standard library or a package already present in this environment. Real projects almost always depend on third-party packages -- and the moment more than one Python project exists on the same machine, a real problem appears: what happens when two different projects need two different, incompatible versions of the same package?

### The problem virtual environments solve

Without any isolation, every project on a machine shares one global set of installed packages. Project A needing `requests` version 2.25 and Project B needing `requests` version 2.31 cannot both be satisfied by a single global installation -- installing one version necessarily means the other project no longer has the version it was built and tested against.

A **virtual environment** is an isolated, self-contained Python installation (technically, a lightweight directory referencing the real interpreter, with its own separate package directory) dedicated to a single project. Packages installed inside it are invisible to every other virtual environment and to the system-wide Python installation.

```text
system Python
     │
     ├── venv for Project A  ──► requests 2.25, flask 2.0
     │
     └── venv for Project B  ──► requests 2.31, django 5.0
```

### Creating and using one

```text
python -m venv .venv
```

creates a new virtual environment in a folder named `.venv`. Activating it (the exact command differs by operating system -- `source .venv/bin/activate` on macOS/Linux, `.venv\Scripts\activate` on Windows) changes the current shell session so that `python` and `pip` refer to the environment's own isolated interpreter and package set, rather than the system-wide ones, for the remainder of that session.

### `pip` and version constraints

Once inside an activated virtual environment, `pip install package_name` installs a package into that environment specifically. Version constraints let you pin exactly what a project depends on:

```text
requests==2.31.0     exactly this version
requests>=2.25,<3.0  any version from 2.25 up to, but not including, 3.0
requests             any version at all -- rarely a good idea for a real project
```

An unconstrained dependency is a genuine risk: a package's newer version, installed later by someone else setting up the same project, might behave differently or even break compatibility outright, and nothing in an unconstrained requirement would warn anyone that the version actually used has silently changed.

### `requirements.txt` -- capturing exactly what a project needs

Intermediate Chapter 64 mentioned `requirements.txt` as part of a project's structure without explaining how it's produced or used. It's a plain text file, one dependency per line, generated from an activated environment's actually-installed packages: 

In [ ]:
requirements_content = '''
requests>=2.25,<3.0
click==8.1.7
python-dotenv>=1.0.0
'''

with open("requirements.txt", "w") as f:
    f.write(requirements_content)

with open("requirements.txt") as f:
    print(f.read())

Anyone else setting up the project runs `pip install -r requirements.txt` inside their own freshly created virtual environment, and `pip` installs exactly the versions specified -- this is what **reproducibility** means in this context: a project's dependencies are recorded explicitly enough that recreating a working environment for it, on a different machine or at a different time, doesn't depend on guessing which versions happened to be installed originally.

### Project metadata -- `pyproject.toml`, briefly

Modern Python projects increasingly describe themselves -- name, version, dependencies, and more -- in a single `pyproject.toml` file, rather than (or alongside) a plain `requirements.txt`. 

In [ ]:
pyproject_content = '''
[project]
name = "my-data-tool"
version = "0.1.0"
description = "A small data processing utility"
dependencies = [
    "requests>=2.25,<3.0",
    "click==8.1.7",
]

[project.optional-dependencies]
dev = ["pytest>=7.0"]
'''

with open("pyproject.toml", "w") as f:
    f.write(pyproject_content)

with open("pyproject.toml") as f:
    print(f.read())

This isn't merely a dependency list -- it's a structured description of the project itself, including a distinction between dependencies the project always needs (`dependencies`) and ones only needed for development, like `pytest` (Intermediate Chapter 62), under `optional-dependencies`. Actually building an installable, distributable package from this file -- the packaging and publishing process itself -- is real, substantial territory belonging to a more specialized, later stage of learning than this chapter aims for; what's worth taking away here is that this file exists, what role it plays, and why a project's identity and dependencies benefit from being declared in one clear, structured place rather than scattered across ad hoc scripts and comments.

### Dependency conflicts

Two packages a project needs can occasionally require incompatible versions of a *third* package they both depend on -- a genuine, sometimes difficult problem with no purely mechanical fix. `pip`'s dependency resolver will refuse to produce an environment with a known conflict, reporting the specific incompatibility rather than silently installing something broken; resolving it typically means finding compatible version ranges for the conflicting packages, or, occasionally, that two specific packages genuinely cannot be used together in a single environment at all, requiring a design decision about the project rather than a purely technical fix.

### One problem

> Write a `requirements.txt` for a hypothetical small web-scraping project needing `requests`, `beautifulsoup4`, and `lxml`, using appropriate version constraints for each (research reasonable version ranges, or use plausible ones), and write a short paragraph, as a comment, explaining why pinning versions matters more for a project meant to be run by other people than for a private, one-off script.

In [ ]:
# TODO: write requirements.txt content and the explanatory comment


## 89. Architecture, Design Patterns, and Maintainable Python

Every chapter in this book has taught a tool. This chapter is about judgment: when to use which tool, and why a program organized thoughtfully stays easy to change, while one organized carelessly gets progressively harder to touch, even if every individual line of it is correct.

### Cohesion and coupling -- the two properties underneath most design advice

**Cohesion** is how strongly related the responsibilities inside a single unit (a function, a class, a module) actually are. **Coupling** is how much one unit depends on the specific internal details of another. Good design, stated in these terms, means high cohesion (each piece has one clear, focused responsibility) and low coupling (pieces interact through narrow, stable interfaces rather than reaching into each other's internals) -- almost every specific piece of design advice that follows in this chapter is a concrete instance of pursuing one or both of these properties.

### A small application, examined honestly

Consider a function that processes an order -- realistic enough to be worth critiquing seriously, not a toy.

In [ ]:
import json


def process_order(order_data):
    order = json.loads(order_data)

    total = 0
    for item in order["items"]:
        total += item["price"] * item["quantity"]

    if order["customer"]["is_premium"]:
        total *= 0.9

    with open("orders.log", "a") as f:
        f.write(f"Order for {order['customer']['name']}: ${total:.2f}\n")

    if total > 1000:
        print(f"ALERT: large order of ${total:.2f}")

    return total

This works, and for a small enough program it might even be reasonable. But it has low cohesion -- parsing JSON, computing a price, applying a discount rule, writing a log entry, and deciding whether to alert on a large order are five genuinely distinct responsibilities, entangled in one function -- and testing any single one of them (Intermediate Chapter 61) means dragging all the others along, since there's no way to test "does the discount calculation work" without also touching the file system and standard output.

### Separating concerns

Restructuring the same behavior around distinct responsibilities, each independently testable: 

In [ ]:
from dataclasses import dataclass
import logging

logger = logging.getLogger(__name__)


@dataclass
class Item:
    price: float
    quantity: int


@dataclass
class Customer:
    name: str
    is_premium: bool


@dataclass
class Order:
    customer: Customer
    items: list


def parse_order(order_data: str) -> Order:
    data = json.loads(order_data)
    items = [Item(price=i["price"], quantity=i["quantity"]) for i in data["items"]]
    customer = Customer(name=data["customer"]["name"], is_premium=data["customer"]["is_premium"])
    return Order(customer=customer, items=items)


def calculate_total(order: Order) -> float:
    total = sum(item.price * item.quantity for item in order.items)
    if order.customer.is_premium:
        total *= 0.9
    return total


def log_order(order: Order, total: float) -> None:
    logger.info(f"Order for {order.customer.name}: ${total:.2f}")
    if total > 1000:
        logger.warning(f"Large order: ${total:.2f}")


def process_order(order_data: str) -> float:
    order = parse_order(order_data)
    total = calculate_total(order)
    log_order(order, total)
    return total

`calculate_total` can now be tested with a plain `Order` object, no JSON, no file I/O, no logging involved at all -- exactly the isolation Intermediate Chapter 61 argued makes tests worth writing in the first place. `parse_order`, `calculate_total`, and `log_order` each have one clear responsibility, and each can change independently -- switching the log destination, or adding a new discount rule, touches exactly one function, not a single sprawling one that does everything.

### Composition over inheritance

Intermediate Part II built a solid case for inheritance as a tool for sharing behavior across related types. It's worth being equally clear about its limits. Inheritance creates a rigid, permanent relationship — a subclass is tightly bound to its base class's implementation details, and changing that base class risks breaking every subclass depending on it, sometimes in ways that aren't obvious from reading the subclass alone. **Composition** — building a class out of other objects it holds and delegates to, rather than inherits from — is frequently the more flexible choice, particularly when the relationship between two types is "has a" or "uses a," rather than genuinely "is a." 

In [ ]:
class EmailNotifier:
    def send(self, message):
        print(f"Emailing: {message}")


class SMSNotifier:
    def send(self, message):
        print(f"Texting: {message}")


class OrderProcessor:
    def __init__(self, notifier):
        self.notifier = notifier

    def process(self, order_total):
        self.notifier.send(f"Order processed: ${order_total:.2f}")


processor = OrderProcessor(EmailNotifier())
processor.process(150.0)

processor_sms = OrderProcessor(SMSNotifier())
processor_sms.process(75.0)

`OrderProcessor` doesn't inherit from either notifier — it holds one, and delegates to it. Swapping notification behavior means passing a different object in, not restructuring an inheritance hierarchy, and `OrderProcessor` works with *any* object exposing a `.send()` method, exactly the duck typing (Intermediate Chapter 40) and `Protocol` (Intermediate Chapter 60) relationship covered previously — composition and structural typing naturally reinforce each other.

### Dependency injection — the same idea, named precisely

What `OrderProcessor` just did — receiving its collaborator (`notifier`) from outside, rather than constructing it internally — is called **dependency injection**. The alternative, `self.notifier = EmailNotifier()` hardcoded inside `__init__`, would tightly couple `OrderProcessor` to one specific notification mechanism, making it impossible to test without actually sending an email, and impossible to reuse with a different notifier without editing `OrderProcessor` itself. Injecting the dependency instead means `OrderProcessor` depends only on the *shape* `notifier` must have (something with `.send()`), not on any specific implementation of it — a direct, practical payoff of low coupling.

### Three patterns, and the actual problems they solve

**Factory** — a function or method whose entire job is constructing an object, particularly when the exact type to construct depends on some input.

In [ ]:
def create_notifier(method):
    if method == "email":
        return EmailNotifier()
    elif method == "sms":
        return SMSNotifier()
    raise ValueError(f"Unknown notification method: {method}")


notifier = create_notifier("sms")
notifier.send("Your order has shipped")

This centralizes the decision of *which concrete class to build* in one place, so the rest of the program never needs to know the full list of notifier types — it only needs to know how to ask for one by name.

**Strategy** — encapsulating an interchangeable algorithm or behavior as an object, so it can be swapped without changing the code that uses it. `OrderProcessor`'s `notifier` is already an example of this: `EmailNotifier` and `SMSNotifier` are two interchangeable *strategies* for the single responsibility "notify someone," selected and injected from outside rather than hardcoded.

**Adapter** — wrapping an object with an incompatible interface so it can be used where a different, expected interface is required.

In [ ]:
class LegacyLogger:
    def write_log(self, text):
        print(f"[legacy] {text}")


class LoggerAdapter:
    def __init__(self, legacy_logger):
        self._legacy = legacy_logger

    def send(self, message):
        self._legacy.write_log(message)


processor_legacy = OrderProcessor(LoggerAdapter(LegacyLogger()))
processor_legacy.process(200.0)

`LegacyLogger` was never designed to satisfy `OrderProcessor`'s expected `.send()` interface, and (being genuinely legacy, third-party, or otherwise off-limits to modify) it can't simply be changed to add one. `LoggerAdapter` bridges the gap — from `OrderProcessor`'s point of view, it's just another object with a `.send()` method, and the mismatch is resolved entirely inside the adapter, without touching either `OrderProcessor` or `LegacyLogger`.

### When a pattern helps, and when it's unnecessary ceremony

Every pattern above solves a specific problem: varying construction logic (factory), varying behavior at runtime (strategy), bridging incompatible interfaces (adapter). Applying one where its specific problem doesn't actually exist adds a layer of indirection with nothing to show for it — a factory function for a type that will only ever be constructed one way is pure ceremony; an adapter wrapping an object that already has the right interface does nothing useful. The right test, before reaching for any of these by name, is the same test this whole chapter has returned to repeatedly: does this change increase cohesion or decrease coupling in a way that will actually matter the next time this code needs to change? If not, the plainer, more direct version was already the better design.

### One problem

> Take the `process_order` example from earlier in this chapter and extend it with a `discount_strategy` parameter (dependency-injected, following the strategy pattern) supporting at least two different discount rules (e.g., percentage-based and flat-amount-based), without `calculate_total` needing an `if`/`elif` chain checking which kind of discount is active.

In [ ]:
from dataclasses import dataclass

# TODO: implement discount strategies and inject one into an order-total calculation


## 90. Advanced Python Capstone — Building a Real Application

This is the final project of the entire three-volume series. Every chapter across Basics, Intermediate, and Advanced has been building toward the ability to design and build something like this independently — no single chapter's tool is enough on its own, and no step-by-step solution is provided, deliberately, for the same reason none was provided for either previous capstone: the value is in making the design decisions yourself.

### The project: a data processing and analysis service

Build a service that ingests structured records (from a file, simulating an external data source), processes and analyzes them, exposes results through a command-line interface, and does all of this in a way that would be reasonable to hand to another developer to maintain.

### Functional requirements

1. **Ingest data** from a JSON or CSV file (Intermediate Chapter 55) containing a batch of records — choose a realistic domain (sales transactions, sensor readings, user events — your choice), each record having at least four meaningful fields, including at least one numeric field suitable for aggregation and one categorical field suitable for grouping.
2. **Validate incoming records**, rejecting malformed ones with a specific, custom exception (Intermediate Chapter 52) rather than crashing the whole batch — the service should report how many records were accepted and how many were rejected, and why.
3. **Compute at least three distinct analyses** over the valid records — for example, a total or average grouped by category, a count of records matching some condition, and a time-based or ordering-based summary. At least one of these analyses should be implemented as a generator pipeline (Chapter 71), processing records lazily rather than materializing every intermediate step as a full list.
4. **Support filtering** records by at least one criterion before analysis, using pattern matching (Chapter 70) somewhere reasonable in the filtering or record-classification logic.
5. **Expose the service through a CLI** (Chapter 87), with at least two subcommands — for instance, `analyze` (run the analyses and print a report) and `validate` (check a file's records without running full analysis).

### Technical requirements

6. **Model records using dataclasses** (Intermediate Chapter 58), with appropriate type hints (Intermediate Chapter 59), including at least one `Optional` or `Union` field where genuinely appropriate.
7. **Design a small class hierarchy** for at least one meaningful part of the domain (different record types, or different analysis strategies via the strategy pattern from Chapter 89), using composition or inheritance deliberately, based on which relationship is actually appropriate — justify the choice in your project's `README.md`.
8. **Use a context manager** for any file or resource handling, writing a custom one (Chapter 50 or 51) somewhere it genuinely adds value beyond what `open()` alone provides — for instance, timing how long ingestion takes, or ensuring a partial output file is cleaned up if processing fails partway through.
9. **Log meaningfully** (Chapter 86) rather than using `print()` for anything beyond the CLI's actual user-facing output — ingestion progress, validation failures, and analysis completion should all be logged at appropriate severity levels.
10. **Read configuration from environment variables** (Chapter 86) for at least one genuinely configurable value — the input file path, or a log level, are both reasonable choices.

### Performance and concurrency

11. **Include at least one measured performance comparison** (Chapter 83) between two implementations of some part of the pipeline — for instance, an eager, list-based version of one analysis versus a lazy, generator-based version — printed or logged as part of the `analyze` subcommand's output, or documented separately with actual measured numbers in the `README.md`.
12. **Use concurrency or async where it's genuinely justified** by the workload, not merely to satisfy this requirement — if your service simulates fetching records from multiple external sources, that's a natural fit for `asyncio` (Part III); if it involves genuinely CPU-heavy analysis across independent chunks of data, that's a natural fit for `multiprocessing` (Chapter 77). Document, in a comment or the `README.md`, why the specific concurrency model you chose fits the actual workload, referencing Chapter 80's decision framework directly.

### Testing, structure, and documentation

13. **Write a `pytest` test suite** (Intermediate Chapter 62) covering record validation (both valid and deliberately invalid input), at least one analysis function in isolation, and the custom exception's behavior.
14. **Organize the project** following Intermediate Chapter 64's structure — a package, a separate `tests/` directory, a `README.md` explaining what the service does, how to run it, and the design decisions requirement 7 asked you to justify.

### What "done" looks like

A person unfamiliar with this project should be able to clone it, read the `README.md`, install its dependencies from a `requirements.txt` or `pyproject.toml` (Chapter 88), run its test suite successfully, and use its CLI to process a sample data file and get a sensible report — without reading a single line of the implementation first. That standard — not "every requirement above is technically present somewhere" — is the actual target.

This project is intentionally larger and less specified than anything earlier in the series. Sketch the architecture before writing code: what are the record types, what does the validation boundary look like, which analyses are genuinely independent of each other, where does logging belong versus where CLI output belongs, and — honestly, based on Chapter 80's framework — whether this particular service actually benefits from concurrency at all, or whether a synchronous implementation is the more honest choice for what it actually does. Every one of those is a real decision this book has given you the tools to make, and none of them has one single correct answer.

In [ ]:
# Your data processing and analysis service goes here.
# Start with an architecture sketch as comments: record types, package layout,
# which analyses are independent, and where logging vs. CLI output belongs.


---

# Solutions

Solutions to every chapter's problem, in order. Each one is written to be read after a genuine attempt, not as a substitute for one — several include a short trace or explanation of the reasoning, not just the final code, exactly as a solutions section in a real technical book should.

### 61 — `Vector2D`

In [ ]:
class Vector2D:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __add__(self, other):
        return Vector2D(self.x + other.x, self.y + other.y)

    def __eq__(self, other):
        return self.x == other.x and self.y == other.y

    def __len__(self):
        return 0 if self.x == 0 and self.y == 0 else 1


v1 = Vector2D(1, 2)
v2 = Vector2D(3, 4)
v3 = Vector2D(4, 6)
print(v1 + v2 == v3)
print(len(Vector2D(0, 0)), len(v1))

Each dunder method here is independent -- `__add__` builds a new object rather than mutating either operand (following the pattern established for immutable-feeling value types since Basics' numbers), `__eq__` compares field by field, and `__len__` maps the object's state onto the specific rule the problem asked for. `v1 + v2 == v3` works because `+` runs first (higher precedence than `==`), producing a new `Vector2D` that `__eq__` then compares against `v3`.

### 62 — Attribute shadowing via `__dict__`

In [ ]:
class Dog:
    species = "Canis familiaris"


rex = Dog()
rex.species = "Very Good Boy"

print(rex.__dict__)
print(Dog.__dict__["species"])
# Lookup finds "species" in rex.__dict__ first, so rex.species reports the instance
# value; Dog.species (accessed through the class directly) is untouched, since the
# instance attribute never modified the class's own copy.
print(rex.species)
print(Dog.species)

### 63 — `ReadOnlyProxy`

In [ ]:
class Target:
    def __init__(self):
        self.value = 42


class ReadOnlyProxy:
    def __init__(self, wrapped):
        object.__setattr__(self, "_wrapped", wrapped)

    def __getattr__(self, name):
        return getattr(self._wrapped, name)

    def __setattr__(self, name, value):
        raise AttributeError("This object is read-only")


proxy = ReadOnlyProxy(Target())
print(proxy.value)
proxy.value = 100

`object.__setattr__(self, "_wrapped", wrapped)` in `__init__` is necessary specifically because the proxy's own `__setattr__` unconditionally raises -- setting `_wrapped` through the ordinary `self._wrapped = wrapped` syntax would trigger that same override and fail immediately. Bypassing it via `object.__setattr__` directly is the one legitimate escape hatch, used only for the proxy's own internal setup.

### 64 — `Typed` descriptor

In [ ]:
class Typed:
    def __init__(self, expected_type):
        self.expected_type = expected_type

    def __set_name__(self, owner, name):
        self.name = "_" + name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return getattr(instance, self.name)

    def __set__(self, instance, value):
        if not isinstance(value, self.expected_type):
            raise TypeError(f"expected {self.expected_type.__name__}, got {type(value).__name__}")
        setattr(instance, self.name, value)


class Person:
    name = Typed(str)
    age = Typed(int)

    def __init__(self, name, age):
        self.name = name
        self.age = age


p = Person("Ali", 25)
print(p.name, p.age)
p.age = "twenty-five" 

### 65 — `SingletonMeta`

In [ ]:
class SingletonMeta(type):
    _instances = {}

    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            cls._instances[cls] = super().__call__(*args, **kwargs)
        return cls._instances[cls]


class Configuration(metaclass=SingletonMeta):
    def __init__(self):
        self.settings = {}


c1 = Configuration()
c2 = Configuration()
c1.settings["debug"] = True

print(c1 is c2)
print(c2.settings)

`__call__` on the metaclass intercepts `Configuration(...)` itself -- calling a class normally invokes `type.__call__`, which creates a new instance every time; overriding `__call__` on the metaclass lets `SingletonMeta` check a cache first, returning the existing instance for any class using it as its metaclass, rather than constructing a fresh one on every call.

### 66 — `make_validators`

In [ ]:
def make_validators(rules):
    validators = {}
    for field, minimum in rules.items():
        def validator(value, minimum=minimum):
            return value >= minimum
        validators[field] = validator
    return validators


validators = make_validators({"age": 18, "score": 60})
print(validators["age"](20), validators["age"](15))
print(validators["score"](75), validators["score"](40))

`minimum=minimum` freezes each iteration's value into an independent default argument, exactly as Chapter 66 demonstrated -- without it, every validator would share the loop's final `minimum` value, the same late-binding trap.

### 67 — `deprecated`

In [ ]:
from functools import wraps


def deprecated(replacement):
    def decorator(func):
        state = {"warned": False}

        @wraps(func)
        def wrapper(*args, **kwargs):
            if not state["warned"]:
                print(f"Warning: {func.__name__} is deprecated, use {replacement} instead")
                state["warned"] = True
            return func(*args, **kwargs)
        return wrapper
    return decorator


@deprecated(replacement="new_add")
def old_add(a, b):
    return a + b


print(old_add(2, 3))
print(old_add(4, 5))

`state["warned"]` (a dictionary used purely to hold a mutable flag) persists across calls via the closure, exactly like Chapter 67's `count_calls`. A dictionary is used instead of a plain variable with `nonlocal` here mainly for brevity, though `nonlocal warned` with a plain boolean would work identically.

### 68 — `describe_function`

In [ ]:
import inspect


def describe_function(func):
    sig = inspect.signature(func)
    for name, param in sig.parameters.items():
        if param.default is inspect.Parameter.empty:
            print(f"{name}: required")
        else:
            print(f"{name}: default = {param.default!r}")


def example(a, b=10, *, c="hello"):
    pass


describe_function(example)

### 69 — `to_json_value`

In [ ]:
from functools import singledispatch


@singledispatch
def to_json_value(value):
    raise TypeError(f"Cannot convert {type(value).__name__} to a JSON value")


@to_json_value.register
def _(value: int):
    return value


@to_json_value.register
def _(value: float):
    return value


@to_json_value.register
def _(value: str):
    return value


@to_json_value.register
def _(value: bool):
    return value


@to_json_value.register
def _(value: list):
    return [to_json_value(v) for v in value]


@to_json_value.register
def _(value: dict):
    return {k: to_json_value(v) for k, v in value.items()}


print(to_json_value({"name": "Ali", "scores": [1, 2, 3], "active": True}))

Registrations for `list` and `dict` recurse back into `to_json_value` for each element, which is what makes this correctly handle arbitrarily nested structures rather than only a single flat level.

### 70 — `process_response`

In [ ]:
def process_response(response):
    match response:
        case {"status": "success", "data": data}:
            return f"Success with {len(data)} item(s)"
        case {"status": "error", "code": code, "message": message}:
            return f"Error {code}: {message}"
        case _:
            return "Unrecognized response shape"


print(process_response({"status": "success", "data": [1, 2, 3]}))
print(process_response({"status": "error", "code": 404, "message": "Not found"}))
print(process_response({"status": "unknown"}))

### 71 — Lazy even Fibonacci numbers

In [ ]:
import itertools


def fibonacci():
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b


def evens(numbers):
    for n in numbers:
        if n % 2 == 0:
            yield n


first_ten_even_fibs = itertools.islice(evens(fibonacci()), 10)
print(list(first_ten_even_fibs))

`fibonacci()` is infinite, exactly like Chapter 71's `integers_from`; `evens` filters lazily; `itertools.islice` stops pulling after ten matches. Nothing downstream of the tenth even Fibonacci number is ever computed.

### 72 — Sequential versus concurrent fetch timing

In [ ]:
import asyncio
import time


async def download(name, delay):
    await asyncio.sleep(delay)
    return f"{name} downloaded"


async def main():
    start = time.perf_counter()
    await download("file1", 1)
    await download("file2", 1)
    await download("file3", 1)
    sequential_time = time.perf_counter() - start

    start = time.perf_counter()
    await asyncio.gather(download("file1", 1), download("file2", 1), download("file3", 1))
    concurrent_time = time.perf_counter() - start

    print(f"sequential: {sequential_time:.2f}s, concurrent: {concurrent_time:.2f}s")


await main()

### 73 — Observing actual resume order

In [ ]:
import asyncio
import time


async def task(name, delay, start_time):
    print(f"{name}: starting")
    await asyncio.sleep(delay)
    print(f"{name}: finished after {time.perf_counter() - start_time:.2f}s")


async def main():
    start_time = time.perf_counter()
    await asyncio.gather(
        task("A", 1.5, start_time),
        task("B", 0.5, start_time),
        task("C", 1.0, start_time),
    )
    # B finishes first (shortest delay), then C, then A -- the event loop resumes
    # whichever coroutine's wait completes first, regardless of the order the
    # tasks were written or started in.


await main()

### 74 — `fetch` with `return_exceptions=True`

In [ ]:
import asyncio


async def fetch(url, delay, fail_for):
    await asyncio.sleep(delay)
    if url == fail_for:
        raise ValueError(f"failed to fetch {url}")
    return f"content from {url}"


async def main():
    urls = ["a.com", "b.com", "c.com", "d.com"]
    results = await asyncio.gather(
        *(fetch(url, 0.2, fail_for="c.com") for url in urls),
        return_exceptions=True
    )
    for url, result in zip(urls, results):
        if isinstance(result, Exception):
            print(f"{url}: FAILED -- {result}")
        else:
            print(f"{url}: {result}")


await main()

### 75 — `paginated_results`

In [ ]:
import asyncio


async def paginated_results(pages, delay):
    for page_num in range(pages):
        await asyncio.sleep(delay)
        yield [f"item-{page_num}-{i}" for i in range(3)]


async def main():
    total_items = 0
    async for page in paginated_results(pages=3, delay=0.2):
        total_items += len(page)
        print(f"received page: {page} (running total: {total_items})")


await main()

### 76 — Threaded I/O-bound vs. CPU-bound comparison

In [ ]:
import threading
import time


def io_task(delay):
    time.sleep(delay)


def cpu_task(n):
    total = 0
    for i in range(n):
        total += i * i
    return total


def time_sequential(func, args_list):
    start = time.perf_counter()
    for args in args_list:
        func(*args)
    return time.perf_counter() - start


def time_threaded(func, args_list):
    start = time.perf_counter()
    threads = [threading.Thread(target=func, args=args) for args in args_list]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    return time.perf_counter() - start


io_args = [(0.5,)] * 4
print(f"I/O sequential: {time_sequential(io_task, io_args):.2f}s")
print(f"I/O threaded:   {time_threaded(io_task, io_args):.2f}s")

cpu_args = [(5_000_000,)] * 4
print(f"CPU sequential: {time_sequential(cpu_task, cpu_args):.2f}s")
print(f"CPU threaded:   {time_threaded(cpu_task, cpu_args):.2f}s")

The I/O comparison shows threading winning clearly; the CPU comparison shows little to no improvement, exactly matching Chapters 76 and 78's explanation of the GIL.

### 77 — Multiprocessing prime check

In [ ]:
import multiprocessing as mp
import time


def is_prime(n):
    if n < 2:
        return False
    for i in range(2, int(n ** 0.5) + 1):
        if n % i == 0:
            return False
    return True


numbers = [100_000_007, 100_000_037, 100_000_039, 100_000_049,
           100_000_073, 100_000_081, 100_000_123, 100_000_127]

start = time.perf_counter()
results_sequential = [is_prime(n) for n in numbers]
print(f"sequential: {time.perf_counter() - start:.2f}s")

start = time.perf_counter()
with mp.Pool(4) as pool:
    results_parallel = pool.map(is_prime, numbers)
print(f"multiprocessing: {time.perf_counter() - start:.2f}s")

print(results_sequential == results_parallel)

### 78 — CPU-bound thread delaying a timer thread

In [ ]:
import threading
import time


def cpu_bound():
    total = 0
    for i in range(100_000_000):
        total += i


def timer():
    for _ in range(5):
        print(time.strftime("%H:%M:%S"))
        time.sleep(1)


t1 = threading.Thread(target=cpu_bound)
t2 = threading.Thread(target=timer)
t1.start()
t2.start()
t1.join()
t2.join()
# The timer thread's printouts are often visibly delayed or irregular while the
# CPU-bound thread runs, because the GIL is only released periodically during
# pure computation (rather than at a clear, deliberate suspension point the way
# an I/O wait releases it), so the timer thread doesn't get as many fair chances
# to run exactly on schedule.

### 79 — Producer-consumer with a shared total

In [ ]:
import threading
import queue
import time

work_queue = queue.Queue()
total_lock = threading.Lock()
total = 0


def producer(start, count):
    for i in range(start, start + count):
        work_queue.put(i)
        time.sleep(0.01)


def consumer(num_producers_done):
    global total
    finished = 0
    while finished < num_producers_done:
        item = work_queue.get()
        if item is None:
            finished += 1
            continue
        with total_lock:
            total += item


producer1 = threading.Thread(target=producer, args=(0, 10))
producer2 = threading.Thread(target=producer, args=(10, 10))
consumer_thread = threading.Thread(target=consumer, args=(2,))

consumer_thread.start()
producer1.start()
producer2.start()
producer1.join()
producer2.join()
work_queue.put(None)
work_queue.put(None)
consumer_thread.join()

print(total)
print("expected:", sum(range(20)))

### 80 — Choosing a concurrency model

In [ ]:
# 1. Hashing ten thousand files on disk: CPU-bound (hashing is pure computation,
#    with disk reads being comparatively fast and often cached). Use multiprocessing --
#    threads would be capped by the GIL for the hashing itself.
#
# 2. Sending push notifications to fifty thousand devices: I/O-bound, dominated by
#    waiting on network responses, at very large scale. Prefer asyncio -- fifty
#    thousand OS threads would be far too heavy, while a single-threaded event loop
#    can hold that many requests in flight cheaply.
#
# 3. A physics simulation updating a shared grid where each cell depends on neighbors'
#    previous values: this has real shared state with tight dependencies between
#    workers, and is CPU-bound. Multiprocessing is still the right family (the
#    GIL rules out threads for CPU-bound speedup), but it requires careful, explicit
#    data partitioning and synchronization between processes each timestep, since
#    processes don't share memory automatically -- worth flagging as the added
#    complexity cost this specific workload's shared state actually creates.

### 81 — `independent_copy`

In [ ]:
def independent_copy(matrix):
    return [[cell for cell in row] for row in matrix]


original = [[1, 2], [3, 4]]
copy = independent_copy(original)
copy[0][0] = 999

print(original)
print(copy)

### 82 — Parent-child cycle versus `weakref`

In [ ]:
import weakref
import gc


class Parent:
    def __init__(self, name):
        self.name = name
        self.children = []
    def __del__(self):
        print(f"Parent {self.name} destroyed")


class ChildStrong:
    def __init__(self, name, parent):
        self.name = name
        self.parent = parent
    def __del__(self):
        print(f"ChildStrong {self.name} destroyed")


class ChildWeak:
    def __init__(self, name, parent):
        self.name = name
        self.parent = weakref.ref(parent)
    def __del__(self):
        print(f"ChildWeak {self.name} destroyed")


print("-- strong reference cycle --")
p1 = Parent("P1")
c1 = ChildStrong("C1", p1)
p1.children.append(c1)
del p1, c1
print("names deleted -- forcing collection:")
gc.collect()

print("-- weak reference, no cycle --")
p2 = Parent("P2")
c2 = ChildWeak("C2", p2)
p2.children.append(c2)
del p2, c2
print("names deleted -- both destroyed immediately, no gc.collect() needed")

### 83 — Prime-check timing comparison

In [ ]:
import timeit
import math


def is_prime_naive(n):
    if n < 2:
        return False
    for i in range(2, n):
        if n % i == 0:
            return False
    return True


def is_prime_sqrt(n):
    if n < 2:
        return False
    for i in range(2, int(math.sqrt(n)) + 1):
        if n % i == 0:
            return False
    return True


numbers = [104729, 104723, 104717, 104711] * 5

naive_time = timeit.timeit(lambda: [is_prime_naive(n) for n in numbers], number=3)
sqrt_time = timeit.timeit(lambda: [is_prime_sqrt(n) for n in numbers], number=3)

print(f"naive: {naive_time:.4f}s, sqrt-bounded: {sqrt_time:.4f}s")

### 84 — Lookup-table versus scanning

In [ ]:
import timeit
import random

orders = [{"customer_id": random.randint(1, 1000), "amount": random.randint(10, 500)}
          for _ in range(20000)]


def find_orders_scan(orders, customer_id):
    return [o for o in orders if o["customer_id"] == customer_id]


def build_index(orders):
    index = {}
    for order in orders:
        index.setdefault(order["customer_id"], []).append(order)
    return index


scan_time = timeit.timeit(lambda: find_orders_scan(orders, 42), number=10)


def indexed_lookup():
    index = build_index(orders)
    return index.get(42, [])


indexed_time = timeit.timeit(indexed_lookup, number=10)

print(f"scanning: {scan_time:.4f}s, building+lookup: {indexed_time:.4f}s")
# Building the index once and reusing it (rather than rebuilding it per lookup,
# as this simplified timing does) is the realistic win -- ten separate lookups
# against a single pre-built index would be dramatically faster than shown here.

### 85 — `process_large_file`

In [ ]:
def batched(iterable, batch_size):
    batch = []
    for item in iterable:
        batch.append(item)
        if len(batch) == batch_size:
            yield batch
            batch = []
    if batch:
        yield batch


def process_large_file(filename, chunk_size):
    with open(filename, "r") as f:
        for batch_num, batch in enumerate(batched(f, chunk_size)):
            avg_length = sum(len(line) for line in batch) / len(batch)
            print(f"batch {batch_num}: average line length {avg_length:.1f}")


process_large_file("large_sample.txt", chunk_size=50000)

### 86 — Configurable logger

In [ ]:
import logging
import os

logger = logging.getLogger("solution86")
logger.setLevel(logging.DEBUG)
logger.handlers.clear()

file_handler = logging.FileHandler("solution86.log")
file_handler.setLevel(logging.DEBUG)

console_level = getattr(logging, os.environ.get("LOG_LEVEL", "WARNING").upper(), logging.WARNING)
console_handler = logging.StreamHandler()
console_handler.setLevel(console_level)

formatter = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")
file_handler.setFormatter(formatter)
console_handler.setFormatter(formatter)

logger.addHandler(file_handler)
logger.addHandler(console_handler)

logger.debug("debug message -- file only, by default")
logger.warning("warning message -- both")

try:
    1 / 0
except ZeroDivisionError:
    logger.exception("caught a division error")

with open("solution86.log") as f:
    print(f.read())

### 87 — CLI with `convert` and `validate` subcommands

In [ ]:
import subprocess

script = '''
import argparse

parser = argparse.ArgumentParser(description="A small file utility")
subparsers = parser.add_subparsers(dest="command", required=True)

convert_parser = subparsers.add_parser("convert")
convert_parser.add_argument("filename")
convert_parser.add_argument("--to", default="json")

validate_parser = subparsers.add_parser("validate")
validate_parser.add_argument("filename")
validate_parser.add_argument("--strict", action="store_true")

args = parser.parse_args()

if args.command == "convert":
    print(f"Converting {args.filename} to {args.to}")
elif args.command == "validate":
    print(f"Validating {args.filename} (strict={args.strict})")
'''

with open("solution87.py", "w") as f:
    f.write(script)

print(subprocess.run(["python", "solution87.py", "convert", "data.csv", "--to", "xml"],
                      capture_output=True, text=True).stdout)
print(subprocess.run(["python", "solution87.py", "validate", "data.csv", "--strict"],
                      capture_output=True, text=True).stdout)
print(subprocess.run(["python", "solution87.py", "--help"],
                      capture_output=True, text=True).stdout)

### 88 — `requirements.txt` for a scraping project

In [ ]:
requirements = '''
requests>=2.31,<3.0
beautifulsoup4>=4.12,<5.0
lxml>=4.9,<5.0
'''
print(requirements)

# Pinning matters far more once other people run the project: an unconstrained
# "requests" could silently resolve to a brand-new major version months from now,
# potentially changing behavior or breaking compatibility, on a machine the
# original author never tested against. For a private, one-off script run only
# by its author, on one known-working machine, that risk is much smaller --
# though even then, pinning costs nothing and avoids a nasty surprise later.

### 89 — Injected discount strategies

In [ ]:
from dataclasses import dataclass


@dataclass
class Item:
    price: float
    quantity: int


@dataclass
class Order:
    items: list


def percentage_discount(rate):
    def strategy(total):
        return total * (1 - rate)
    return strategy


def flat_discount(amount):
    def strategy(total):
        return max(0, total - amount)
    return strategy


def calculate_total(order, discount_strategy=None):
    subtotal = sum(item.price * item.quantity for item in order.items)
    if discount_strategy is not None:
        return discount_strategy(subtotal)
    return subtotal


order = Order(items=[Item(price=50, quantity=2)])

print(calculate_total(order))
print(calculate_total(order, discount_strategy=percentage_discount(0.1)))
print(calculate_total(order, discount_strategy=flat_discount(15)))

`calculate_total` never inspects what *kind* of discount it received -- it just calls `discount_strategy(subtotal)` if one was given, exactly the strategy pattern from Chapter 89. Adding a new discount rule means writing one more small factory function, never touching `calculate_total` itself.

### 90 — Capstone

No single solution is provided for the capstone, matching the same policy the Basics and Intermediate capstones followed. Its value is in the design decisions you make yourself, checked against the requirements stated in Chapter 90, not against a hidden reference implementation.

---

## Where the series leaves you

Three volumes, ninety chapters, one continuous thread: understand what Python is actually doing, not merely what syntax produces what output. Basics built the vocabulary. Intermediate built the object model and the developer's everyday toolkit. This volume opened up the machinery underneath both — attribute lookup, the descriptor protocol, coroutines and the event loop, the GIL, reference counting, and the discipline of measuring before optimizing.

None of this was ever the destination. It's the foundation for reading code you didn't write, debugging behavior that doesn't match your first guess, and making deliberate, defensible decisions about how to build something — which is the actual, ongoing work of being a developer, long after any curriculum ends.

---

**Code With SUZH**

Understand the behavior.
Write the code.
Build with Python.

**SUZH TEAM**